<a href="https://colab.research.google.com/github/ilyaaannn/justchill/blob/main/MODEL_3_kategori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install kagglehub scikit-learn pandas numpy joblib imblearn matplotlib seaborn scikit-fuzzy

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier, BallTree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import joblib
import os
import shutil
import warnings as warning
warning.filterwarnings("ignore")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

SEED = 42

# ====== LOAD DATASET 1: SENSOR-BASED AQUAPONICS ======
print("\nLOADING DATASET 1: Sensor-Based Aquaponics Fish Pond")
try:
    df1 = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "samahfetouh/cleaned-aquaponics-pond-dataset",
        "IoTPond3.csv"
    )
    print(f"Dataset 1 loaded successfully: {df1.shape[0]} rows, {df1.shape[1]} columns")
    print(f"   Columns: {list(df1.columns)}")
except Exception as e:
    print(f"❌ Error loading dataset 1: {e}")
    raise

# ====== LOAD DATASET 2: SMALL AQUACULTURE FISHPOND ======
print("\nLOADING DATASET 2: Small Aquaculture Fishpond")
try:
    df2 = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "bobsis/small-aquaculture-fishpond",
        "pond_iot_2023_raw.csv"
    )
    print(f"Dataset 2 loaded successfully: {df2.shape[0]} rows, {df2.shape[1]} columns")
    print(f"   Columns: {list(df2.columns)}")
except Exception as e:
    print(f"❌ Error loading dataset 2: {e}")
    raise

# ====== METODE DATA FUSION: GROUPBY MEAN MAPPING (PERBAIKAN) ======
print("\n⚙  Melakukan Proses Data Fusion Berbasis Jangkar Suhu...")

# 1. Ekstrak kolom dan bersihkan nilai kosong (dropna)
df1_extracted = df1[['disolved_oxg', 'temperature']].dropna().copy()
df1_extracted.columns = ['DO(mg/l)', 'Temp_Jembatan']

# PEMBERSIHAN OUTLIER: Hapus nilai DO ekstrem (> 15 mg/l) yang tidak masuk akal secara biologis
# Langkah ini memperbaiki anomali nilai DO 34.47 mg/l yang sempat muncul
df1_extracted = df1_extracted[df1_extracted['DO(mg/l)'] <= 15.0]

df2_extracted = df2[['water_pH', 'TDS', 'water_temp']].dropna().copy()
df2_extracted.columns = ['pH(ph_units)', 'TDS(ppm)', 'Temp_Jembatan']

print(f"✓ Dataset 1 (setelah pembersihan outlier): {len(df1_extracted):,} baris")
print(f"✓ Dataset 2 (setelah dropna): {len(df2_extracted):,} baris")

# 2. Bulatkan nilai suhu ke 1 desimal agar proses sinkronisasi akurat
df1_extracted['Temp_Jembatan'] = df1_extracted['Temp_Jembatan'].round(1)
df2_extracted['Temp_Jembatan'] = df2_extracted['Temp_Jembatan'].round(1)

# 3. CRUCIAL FIX: Cari rata-rata nilai DO untuk setiap nilai derajat suhu unik di Dataset 1
# Langkah ini menjamin TIDAK AKAN ADA baris terduplikasi/kembar di hasil akhir
kamus_do_rata2 = df1_extracted.groupby('Temp_Jembatan')['DO(mg/l)'].mean().to_dict()

# 4. Suntikkan nilai DO ke Dataset 2 berdasarkan kesamaan suhunya
df2_extracted['DO(mg/l)'] = df2_extracted['Temp_Jembatan'].map(kamus_do_rata2)

# 5. Hapus baris data yang suhunya tidak memiliki pasangan di Kamus DO (NaN)
df3_combined = df2_extracted.dropna().reset_index(drop=True)

# 6. Atur ulang susunan kolom agar rapi sesuai struktur awal programmu
df3_combined = df3_combined[['DO(mg/l)', 'pH(ph_units)', 'TDS(ppm)', 'Temp_Jembatan']]
df3_combined.columns = ['DO(mg/l)', 'pH(ph_units)', 'TDS(ppm)', 'Temp(cel)']

# 7. EFEKTIVITAS DATA: Batasi jumlah baris ke angka paling optimal untuk performa cepat KNN
MAX_ROWS = 10000
if len(df3_combined) > MAX_ROWS:
    df3_combined = df3_combined.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f"✓ Dataset gabungan dibatasi ke {MAX_ROWS:,} baris untuk efisiensi RAM & kecepatan real-time.")

print(f"\n✓ Dataset 3 Berhasil Dibuat! (Data Fusion Tanpa Duplikasi)")
print(f"    ✓ Shape      : {df3_combined.shape[0]:,} baris × {df3_combined.shape[1]} columns")
print(f"    ✓ Columns    : {list(df3_combined.columns)}")
print(f"    ✓ Missing values per kolom:")
for col in df3_combined.columns:
    print(f"       - {col}: {df3_combined[col].isnull().sum()}")

# ----- Verifikasi konsistensi ilmiah: DO vs Suhu -----
corr_do_temp = df3_combined['DO(mg/l)'].corr(df3_combined['Temp(cel)'])
print(f"\n✓ Validasi Ilmiah — Korelasi DO ↔ Suhu: {corr_do_temp:.4f}")
if corr_do_temp < 0:
    print("   ✅ Korelasi negatif sesuai teori limnologi (suhu ↑ → DO ↓)")
else:
    print("   ⚠  Korelasi positif/mendekati nol. Namun data aman dari duplikasi.")

# ----- Simpan sebagai dataframe utama -----
df = df3_combined.copy()

print(f"\n✓ Dataframe final (df) siap masuk ke proses Fuzzy dan KNN.")
print()
print(df.info())
print(df.describe())

## **METODE PERTAMA:** Rule-based counting

In [ ]:
# Analisis statistik Rule-based counting
print("\nStatistical Summary:")
print(df.describe())

def classify_water_quality(row):
    bahaya_count = 0
    normal_count = 0
    ideal_count = 0

    # check pH
    if 'pH' in row or 'water_pH' in row or 'pH(ph_units)' in row:
        pH = row.get ('pH(ph_units)', row.get('water_pH'))
        if 6.8 <= pH <= 7.2:
            ideal_count += 1
        elif pH < 6.0 or pH > 8.0:
            bahaya_count += 1
        else:
            normal_count += 1

    # check temperature
    if 'temperature' in row or 'Temp(cel)' in row or 'water_temp' in row:
        temp = row.get('Temp(cel)', row.get('water_temp'))
        if 25.0 <= temp <= 29.0:
            ideal_count += 1
        elif temp < 22.0 or temp > 32.0:
            bahaya_count += 1
        else:
            normal_count += 1

    # check TDS (Total Dissolved Solids)
    if 'TDS(ppm)' in row or 'TDS' in row:
        TDS = row.get('TDS(ppm)', row.get('TDS'))
        if 150 <= TDS <= 400:
            ideal_count += 1
        elif TDS > 1000:
            bahaya_count += 1
        else:
            normal_count += 1

    # check DO (Dissolved Oxygen)
    if 'do' in row or 'DO' in row or 'DO(mg/l)' in row:
        DO = row.get('DO(mg/l)', row.get('DO'))
        if DO > 6.00:
            ideal_count += 1
        elif DO < 3.00:
            bahaya_count += 1
        else:
            normal_count += 1

    # Tentukan status
    if bahaya_count >= 1:
        return 'bahaya'
    elif ideal_count == 4:
        return 'ideal'
    else:
        return 'normal'

# Tambahkan kolom status
df['status'] = df.apply(classify_water_quality, axis=1)

# Definisikan urutan dari TERBAIK ke TERBURUK
order_from_best_to_worst = ['ideal', 'normal', 'bahaya']
status_counts = df['status'].value_counts()
status_counts_ordered = status_counts.reindex(order_from_best_to_worst, fill_value=0)
print("\nDistribusi Status (Terbaik → Terburuk):")
print(status_counts_ordered)

# Pilih fitur untuk training (sesuaikan dengan kolom di dataset)
feature_columns = [col for col in df.columns if col not in ['id','created_date','entry_id','created_at','status']]
print(f"\nFitur yang digunakan: {feature_columns}")

In [ ]:
# ====== BALANCING DATASET SECARA DINAMIS BERBASIS MEDIAN ======
print("\n" + "="*60)
print("⚖️  MENYEIMBANGKAN DATASET (DINAMIS - BERDASARKAN MEDIAN)")
print("="*60)

X = df[feature_columns].values
y = df['status'].values
urutan_label = ['ideal', 'normal', 'bahaya']

# Tampilkan distribusi awal
print("\n📊 Distribusi Kelas SEBELUM Balancing:")
initial_distribution = {}
for label in urutan_label:
    initial_distribution[label] = np.sum(y == label)

for label in urutan_label:
    count = initial_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y):5d} sampel")

# Hitung median sebagai target jumlah sampel per kelas
jumlah_per_kelas = [initial_distribution[label] for label in urutan_label]
target_median = int(np.median(jumlah_per_kelas))

print(f"\n🎯 Median jumlah sampel per kelas: {target_median}")
print("📊 Strategi Dinamis:")
print("   - Kelas < median → oversample")
print("   - Kelas > median → undersample")
print("   - Kelas = median → pertahankan")

# Buat strategi sampling
over_strategy = {}
under_strategy = {}

for label in urutan_label:
    count = initial_distribution[label]
    if count < target_median:
        over_strategy[label] = target_median
    elif count > target_median:
        under_strategy[label] = target_median
    # Jika sama, tidak perlu sampling

# Bangun pipeline hanya jika diperlukan
steps = []

# Tambahkan oversampling jika ada kelas minority
if over_strategy:
    steps.append(('oversample', SMOTE(
        sampling_strategy=over_strategy,
        random_state=42,
        k_neighbors=min(3, min(initial_distribution.values()))  # hindari error jika sampel < 3
    )))

# Tambahkan undersampling jika ada kelas majority
if under_strategy:
    steps.append(('undersample', RandomUnderSampler(
        sampling_strategy=under_strategy,
        random_state=42
    )))

if steps:
    pipeline = Pipeline(steps)
    print("\n⏳ Proses balancing dinamis...")
    X_balanced, y_balanced = pipeline.fit_resample(X, y)
else:
    print("\n⚠️  Tidak diperlukan balancing (semua kelas sudah seimbang).")
    X_balanced, y_balanced = X.copy(), y.copy()

# Tampilkan distribusi setelah balancing
print("\n✅ Distribusi Kelas SETELAH Balancing:")
final_distribution = {}
for label in urutan_label:
    final_distribution[label] = np.sum(y_balanced == label)

for label in urutan_label:
    count = final_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y_balanced)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y_balanced):5d} sampel")

# Shuffle dataset agar random
shuffle_idx = np.random.RandomState(42).permutation(len(y_balanced))
X_balanced = X_balanced[shuffle_idx]
y_balanced = y_balanced[shuffle_idx]

print("\n🔀 Dataset telah di-shuffle untuk memastikan randomness")
print(f"✅ Dataset balanced siap digunakan: {len(y_balanced)} sampel")

# Visualisasi perbandingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before balancing
axes[0].bar(urutan_label, [initial_distribution.get(l, 0) for l in urutan_label],
            color=['green', 'orange', 'red'], alpha=0.7)
axes[0].set_title('Distribusi SEBELUM Balancing', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah Sampel', fontsize=12)
axes[0].set_xticklabels(urutan_label, rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# After balancing
axes[1].bar(urutan_label, [final_distribution.get(l, 0) for l in urutan_label],
            color=['green', 'orange', 'red'], alpha=0.7)
axes[1].set_title('Distribusi SETELAH Balancing', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Jumlah Sampel', fontsize=12)
axes[1].set_xticklabels(urutan_label, rotation=45)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
axes[1].axhline(y=target_median, color='blue', linestyle='--',
                label=f'Median Target: {target_median}', linewidth=2)
axes[1].legend()

plt.tight_layout()
plt.show()

# Update variabel X dan y dengan data yang sudah balanced
X = X_balanced
y = y_balanced

In [ ]:
# ====== 1. PERSIAPAN DATA (SETELAH BALANCING) ======
print("="*50)
print("📊 PERSIAPAN DATA - TRAIN TEST SPLIT")
print("="*50)

urutan_label = ['ideal', 'normal', 'bahaya']
# Visualisasi distribusi kelas setelah balancing
plt.figure(figsize=(8, 5))
status_counts = pd.Series(y_balanced).value_counts()
status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
colors = ['green', 'orange', 'red']
bars = plt.bar(urutan_label, status_counts_ordered, color=colors, alpha=0.7, edgecolor='black')

# Tambahkan nilai di atas bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.title('Distribusi Kelas Kualitas Air (Setelah Balancing)', fontsize=14, fontweight='bold')
plt.ylabel('Jumlah', fontsize=12)
plt.xlabel('Status Kualitas Air', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Split data menjadi training dan testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f"\n✅ Data Training: {len(X_train)} sampel ({len(X_train)/len(X_balanced)*100:.1f}%)")
print(f"✅ Data Testing: {len(X_test)} sampel ({len(X_test)/len(X_balanced)*100:.1f}%)")

# Normalisasi data menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("✅ Data berhasil dibagi dan dinormalisasi (StandardScaler)")

# ====== VALIDASI SILANG K-FOLD ======
print("\n" + "="*50)
print("🔍 TUNING HYPERPARAMETER (K-Nearest Neighbors)")
print("="*50)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_values = list(range(3, 16, 2))  # Perluas rentang pencarian
cv_scores = []
cv_stds = []

print("⏳ Mencari nilai K optimal dengan 5-Fold Cross Validation...\n")

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='euclidean')
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    mean_score = scores.mean()
    std_score = scores.std()

    cv_scores.append(mean_score)
    cv_stds.append(std_score)

    print(f"K={k:2d} | Mean CV Accuracy = {mean_score:.4f} | Std = {std_score:.4f}")

best_k = k_values[np.argmax(cv_scores)]
best_acc = max(cv_scores)

print(f"\n🎯 Nilai K Optimal: {best_k}")
print(f"🎯 Best CV Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

# ====== EVALUASI MODEL ======
print("\n" + "="*50)
print("🔍 EVALUASI MODEL (Test Set)")
print("="*50)

best_model = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='euclidean')
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)
weighted_f1 = f1_score(y_test, y_pred, average='weighted')
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n🚀 TRAINING MODEL DENGAN K={best_k}")
print(f"🎯 Test Accuracy      = {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"🎯 Weighted F1-Score  = {weighted_f1:.4f}")
print(f"🎯 Macro F1-Score     = {macro_f1:.4f}")

# Classification Report (dengan urutan label yang benar)
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, labels=urutan_label, target_names=urutan_label, zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
print("\nActual \\ Predicted:", end="")
for label in urutan_label:
    print(f"{label:>12}", end="")
print()
print("-" * 80)
for i, label in enumerate(urutan_label):
    print(f"{label:>18}", end="")
    for j in range(len(urutan_label)):
        print(f"{cm[i][j]:>12}", end="")
    print()

# ====== ANALISIS LEARNING CURVE ======
print("\n" + "="*50)
print("📈 ANALISIS LEARNING CURVE")
print("="*50)

train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', color='blue', linewidth=2,
         markersize=8, label='Training Score')
plt.plot(train_sizes, test_mean, 'o-', color='green', linewidth=2,
         markersize=8, label='Cross-validation Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.15, color='blue')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std,
                 alpha=0.15, color='green')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Training Examples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - KNN Classifier (Data Balanced)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

# Analisis hasil learning curve
print("\n🔍 Analisis Learning Curve:")
gap = train_mean[-1] - test_mean[-1]
print(f"  - Training Score (final): {train_mean[-1]:.4f}")
print(f"  - CV Score (final):       {test_mean[-1]:.4f}")
print(f"  - Gap:                    {gap:.4f}")

if test_mean[-1] > 0.9 and train_mean[-1] > 0.9 and gap < 0.05:
    performance = "✅ Model terlihat SANGAT BAIK (high performance, low overfitting)"
elif gap > 0.1:
    performance = "⚠️ Model mungkin OVERFITTING (training score >> CV score)"
elif train_mean[-1] < 0.8:
    performance = "⚠️ Model mungkin UNDERFITTING (perlu model lebih kompleks)"
elif test_mean[-1] > 0.85:
    performance = "✅ Model memiliki performa BAIK dan SEIMBANG"
else:
    performance = "⚠️ Model memiliki performa CUKUP (bisa ditingkatkan)"

print(f"  Kesimpulan: {performance}")

In [ ]:
# 📁 Tentukan folder penyimpanan
output_dir = '/content/drive/MyDrive/hasil_machine_learning/3-Rule-Based-counting/'
os.makedirs(output_dir, exist_ok=True)

# ====== SIMPAN SEMUA OUTPUT ======
print("\n" + "="*60)
print("💾 PENYIMPANAN HASIL KE GOOGLE DRIVE")
print("="*60)

# 1. Simpan Model KNN
model_path = os.path.join(output_dir, 'knn_water_quality_model.pkl')
joblib.dump(best_model, model_path)
print(f"✅ Model KNN disimpan: {model_path}")

# 2. Simpan Scaler
scaler_path = os.path.join(output_dir, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"✅ Scaler disimpan: {scaler_path}")

# 3. Simpan Feature Columns
features_path = os.path.join(output_dir, 'feature_columns.pkl')
joblib.dump(feature_columns, features_path)
print(f"✅ Feature columns disimpan: {features_path}")

# 4. Simpan Dataset dengan Status
dataset_path = os.path.join(output_dir, 'dataset_balanced_with_status.csv')
df_balanced = pd.DataFrame(X_balanced, columns=feature_columns)
numeric_columns = ['pH(ph_units)', 'Temp(cel)', 'TDS(ppm)', 'DO(mg/l)']
for col in numeric_columns:
    if col in df_balanced.columns:
        df_balanced[col] = df_balanced[col].astype(float).round(2)
df_balanced['status'] = y_balanced
df_balanced.to_csv(dataset_path, index=False)
print(f"✅ Dataset balanced disimpan: {dataset_path}")

# 5. Simpan Dataset Original dengan Status
dataset_original_path = os.path.join(output_dir, 'dataset_original_with_status.csv')
df.to_csv(dataset_original_path, index=False)
print(f"✅ Dataset original disimpan: {dataset_original_path}")

# 6. Simpan Metadata Model
metadata_path = os.path.join(output_dir, 'model_metadata.txt')
with open(metadata_path, 'w', encoding='utf-8') as f:
    f.write("="*60 + "\n")
    f.write("MODEL METADATA - KNN WATER QUALITY CLASSIFIER\n")
    f.write("="*60 + "\n\n")

    f.write("📊 INFORMASI DATASET\n")
    f.write("-" * 60 + "\n")
    f.write(f"Total Data (Original):        {len(df)} sampel\n")
    f.write(f"Total Data (After Balancing): {len(y_balanced)} sampel\n")
    f.write(f"Training Set:                 {len(y_train)} sampel ({len(y_train)/len(y_balanced)*100:.1f}%)\n")
    f.write(f"Testing Set:                  {len(y_test)} sampel ({len(y_test)/len(y_balanced)*100:.1f}%)\n")
    f.write(f"Jumlah Fitur:                 {len(feature_columns)}\n")
    f.write(f"Nama Fitur:                   {', '.join(feature_columns)}\n\n")

    f.write("🎯 DISTRIBUSI KELAS (BEFORE BALANCING)\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    status_counts = df['status'].value_counts()
    status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
    for label in urutan_label:
        count = status_counts_ordered[label]
        f.write(f"  {label:12s}: {count:5d} ({count/len(df)*100:5.2f}%)\n")
    f.write("\n")

    f.write("🎯 DISTRIBUSI KELAS (AFTER BALANCING)\n")
    f.write("-" * 60 + "\n")
    for label in urutan_label:
        count = np.sum(y_balanced == label)
        f.write(f"  {label:12s}: {count:5d} ({count/len(y_balanced)*100:5.2f}%)\n")
    f.write("\n")

    f.write("🔍 PARAMETER MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Algoritma:                    K-Nearest Neighbors (KNN)\n")
    f.write(f"Nilai K Optimal:              {best_k}\n")
    f.write(f"Weights:                      distance\n")
    f.write(f"Metric:                       euclidean\n")
    f.write(f"Normalisasi:                  StandardScaler\n\n")

    f.write("📈 PERFORMA MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Best CV Accuracy:             {best_acc:.4f} ({best_acc*100:.2f}%)\n")
    f.write(f"Test Accuracy:                {test_accuracy:.4f} ({test_accuracy*100:.2f}%)\n")
    f.write(f"Weighted F1-Score:            {weighted_f1:.4f}\n")
    f.write(f"Macro F1-Score:               {macro_f1:.4f}\n\n")

    f.write("📊 AKURASI PER KELAS\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
    for i, label in enumerate(urutan_label):
        class_correct = cm[i, i]
        class_total = cm[i, :].sum()
        class_accuracy = class_correct / class_total if class_total > 0 else 0
        f.write(f"  {label:12s}: {class_accuracy:.4f} ({class_accuracy*100:.2f}%) - {class_correct}/{class_total} benar\n")
    f.write("\n")

    f.write("📋 CLASSIFICATION REPORT\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    f.write(classification_report(y_test, y_pred, labels=urutan_label, target_names=urutan_label, zero_division=0))
    f.write("\n")

    f.write("🔬 CONFUSION MATRIX\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    f.write("                              Predicted\n")
    f.write("Actual        " + "  ".join([f"{l:8s}" for l in urutan_label]) + "\n")
    for i, label in enumerate(urutan_label):
        f.write(f"{label:9s} " + "  ".join([f"{cm[i,j]:8d}" for j in range(len(urutan_label))]) + "\n")
    f.write("\n")

print(f"✅ Metadata model disimpan: {metadata_path}")

# Simpan Visualisasi
# Visualisasi 1: Distribusi Kelas Balanced
plt.figure(figsize=(8, 5))
status_counts = pd.Series(y_balanced).value_counts()
status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
colors = ['green', 'orange', 'red']
bars = plt.bar(urutan_label, status_counts_ordered, color=colors, alpha=0.7, edgecolor='black')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.title('Distribusi Kelas Kualitas Air (Setelah Balancing)', fontsize=14, fontweight='bold')
plt.ylabel('Jumlah', fontsize=12)
plt.xlabel('Status Kualitas Air', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'class_distribution_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik distribusi disimpan")

# Visualisasi 2: KNN Cross-Validation Results
plt.figure(figsize=(10, 6))
plt.errorbar(k_values, cv_scores, yerr=cv_stds, fmt='o-', capsize=5,
             color='blue', ecolor='red', linewidth=2, markersize=8)
plt.axhline(y=max(cv_scores), color='green', linestyle='--', alpha=0.7, linewidth=2)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Nilai K', fontsize=12)
plt.ylabel('Accuracy (5-Fold CV)', fontsize=12)
plt.title('Performa KNN terhadap Nilai K (Data Balanced)', fontsize=14, fontweight='bold')
plt.xticks(k_values)
plt.scatter(best_k, best_acc, color='red', s=200, zorder=5,
            label=f'Best K={best_k} (Acc={best_acc:.4f})', marker='*')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'knn_cv_results_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik CV results disimpan")

# Visualisasi 3: Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=urutan_label, yticklabels=urutan_label,
            annot_kws={"size": 14, "weight": "bold"},
            cbar_kws={'label': 'Jumlah Sampel'},
            linewidths=1, linecolor='gray')
plt.xlabel('Predicted', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=14, fontweight='bold')
plt.title('Confusion Matrix (Test Set - Data Balanced)', fontsize=16, fontweight='bold')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'confusion_matrix_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik confusion matrix disimpan")

# Visualisasi 4: Learning Curve
train_sizes_lc, train_scores_lc, test_scores_lc = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)
train_mean_lc = np.mean(train_scores_lc, axis=1)
train_std_lc = np.std(train_scores_lc, axis=1)
test_mean_lc = np.mean(test_scores_lc, axis=1)
test_std_lc = np.std(test_scores_lc, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes_lc, train_mean_lc, 'o-', color='blue', linewidth=2,
         markersize=8, label='Training Score')
plt.plot(train_sizes_lc, test_mean_lc, 'o-', color='green', linewidth=2,
         markersize=8, label='Cross-validation Score')
plt.fill_between(train_sizes_lc, train_mean_lc - train_std_lc, train_mean_lc + train_std_lc,
                 alpha=0.15, color='blue')
plt.fill_between(train_sizes_lc, test_mean_lc - test_std_lc, test_mean_lc + test_std_lc,
                 alpha=0.15, color='green')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Training Examples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - KNN Classifier (Data Balanced)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'learning_curve_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik learning curve disimpan")


# Ringkasan File yang Tersimpan
print("\n" + "="*60)
print("📁 DAFTAR FILE TERSIMPAN")
print("="*60)
saved_files = [
    ('Model KNN', 'knn_water_quality_model.pkl'),
    ('Scaler', 'scaler.pkl'),
    ('Feature Columns', 'feature_columns.pkl'),
    ('Dataset Balanced', 'dataset_balanced_with_status.csv'),
    ('Dataset Original', 'dataset_original_with_status.csv'),
    ('Metadata', 'model_metadata.txt'),
    ('Grafik Distribusi', 'class_distribution_balanced.png'),
    ('Grafik CV Results', 'knn_cv_results_balanced.png'),
    ('Grafik Confusion Matrix', 'confusion_matrix_balanced.png'),
    ('Grafik Learning Curve', 'learning_curve_balanced.png')
]

for name, filename in saved_files:
    filepath = os.path.join(output_dir, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        size_str = f"{size/1024:.2f} KB" if size < 1024*1024 else f"{size/(1024*1024):.2f} MB"
        print(f"✅ {name:25s}: {filename:40s} ({size_str})")
    else:
        print(f"❌ {name:25s}: {filename:40s} (TIDAK DITEMUKAN)")

## **METODE KEDUA:** Rule-based priority

In [ ]:
# Analisis statistiK Rule-based prioritas
print("\nStatistical Summary:")
print(df.describe())

def classify_water_quality(row):
  prioritas_order = ['bahaya', 'normal', 'ideal']
  kondisi_terbaik = 'ideal'

  # check PH
  if 'pH' in row or 'water_pH' in row or 'pH(ph_units)' in row:
      pH = row.get ('pH(ph_units)', row.get('water_pH'))
      if pd.notna(pH):
          if pH < 6.0 or pH > 8.0:
              kondisi = 'bahaya'
          elif (6.0 <= pH < 6.8) or (7.2 < pH <= 8.0):
              kondisi = 'normal'
          elif 6.8 <= pH <= 7.2:
              kondisi = 'ideal'
          else:
              kondisi = 'normal'
          if prioritas_order.index(kondisi) < prioritas_order.index(kondisi_terbaik):
                kondisi_terbaik = kondisi

  # check suhu
  if 'temperature' in row or 'Temp(cel)' in row or 'water_temp' in row:
      temp = row.get('Temp(cel)', row.get('water_temp'))
      if pd.notna(temp):
          if temp < 22.0 or temp > 32.0:
              kondisi = 'bahaya'
          elif (22.0 <= temp < 25.0) or (29.0 < temp <= 32.0):
              kondisi = 'normal'
          elif 25.0 <= temp <= 29.0:
              kondisi = 'ideal'
          else:
              kondisi = 'normal'
          if prioritas_order.index(kondisi) < prioritas_order.index(kondisi_terbaik):
                kondisi_terbaik = kondisi

  # check TDS (Total Dissolved Solids)
  if 'TDS(ppm)' in row or 'TDS' in row:
      TDS = row.get('TDS(ppm)', row.get('TDS'))
      if pd.notna(TDS):
          if TDS > 1000:
              kondisi = 'bahaya'
          elif 401 <= TDS <= 1000:
              kondisi = 'normal'
          elif 150 <= TDS <= 400:
              kondisi = 'ideal'
          else:
              kondisi = 'normal'
          if prioritas_order.index(kondisi) < prioritas_order.index(kondisi_terbaik):
                kondisi_terbaik = kondisi

  # check DO (Dissolved Oxygen)
  if 'do' in row or 'DO' in row or 'DO(mg/l)' in row:
      DO = row.get('DO(mg/l)', row.get('DO'))
      if pd.notna(DO):
          if DO < 3.00:
              kondisi = 'bahaya'
          elif 3.00 <= DO <= 6.00:
              kondisi = 'normal'
          elif DO > 6.00:
              kondisi = 'ideal'
          else:
              kondisi = 'normal'
          if prioritas_order.index(kondisi) < prioritas_order.index(kondisi_terbaik):
                kondisi_terbaik = kondisi

  return kondisi_terbaik

# Tambahkan kolom status
df['status'] = df.apply(classify_water_quality, axis=1)

# Definisikan urutan dari TERBAIK ke TERBURUK
order_from_best_to_worst = ['ideal', 'normal', 'bahaya']
status_counts = df['status'].value_counts()
status_counts_ordered = status_counts.reindex(order_from_best_to_worst, fill_value=0)
print("\nDistribusi Status (Terbaik → Terburuk):")
print(status_counts_ordered)

# Pilih fitur untuk training (sesuaikan dengan kolom di dataset)
feature_columns = [col for col in df.columns if col not in ['id','created_date','entry_id','created_at','status']]
print(f"\nFitur yang digunakan: {feature_columns}")

In [ ]:
# ====== BALANCING DATASET SECARA DINAMIS BERBASIS MEDIAN ======
print("\n" + "="*60)
print("⚖️  MENYEIMBANGKAN DATASET (DINAMIS - BERDASARKAN MEDIAN)")
print("="*60)

X = df[feature_columns].values
y = df['status'].values
urutan_label = ['ideal', 'normal', 'bahaya']

# Tampilkan distribusi awal
print("\n📊 Distribusi Kelas SEBELUM Balancing:")
initial_distribution = {}
for label in urutan_label:
    initial_distribution[label] = np.sum(y == label)

for label in urutan_label:
    count = initial_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y):5d} sampel")

# Hitung median sebagai target jumlah sampel per kelas
jumlah_per_kelas = [initial_distribution[label] for label in urutan_label]
target_median = int(np.median(jumlah_per_kelas))

print(f"\n🎯 Median jumlah sampel per kelas: {target_median}")
print("📊 Strategi Dinamis:")
print("   - Kelas < median → oversample")
print("   - Kelas > median → undersample")
print("   - Kelas = median → pertahankan")

# Buat strategi sampling
over_strategy = {}
under_strategy = {}

for label in urutan_label:
    count = initial_distribution[label]
    if count < target_median:
        over_strategy[label] = target_median
    elif count > target_median:
        under_strategy[label] = target_median
    # Jika sama, tidak perlu sampling

# Bangun pipeline hanya jika diperlukan
steps = []

# Tambahkan oversampling jika ada kelas minority
if over_strategy:
    steps.append(('oversample', SMOTE(
        sampling_strategy=over_strategy,
        random_state=42,
        k_neighbors=min(3, min(initial_distribution.values()))  # hindari error jika sampel < 3
    )))

# Tambahkan undersampling jika ada kelas majority
if under_strategy:
    steps.append(('undersample', RandomUnderSampler(
        sampling_strategy=under_strategy,
        random_state=42
    )))

if steps:
    pipeline = Pipeline(steps)
    print("\n⏳ Proses balancing dinamis...")
    X_balanced, y_balanced = pipeline.fit_resample(X, y)
else:
    print("\n⚠️  Tidak diperlukan balancing (semua kelas sudah seimbang).")
    X_balanced, y_balanced = X.copy(), y.copy()

# Tampilkan distribusi setelah balancing
print("\n✅ Distribusi Kelas SETELAH Balancing:")
final_distribution = {}
for label in urutan_label:
    final_distribution[label] = np.sum(y_balanced == label)

for label in urutan_label:
    count = final_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y_balanced)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y_balanced):5d} sampel")

# Shuffle dataset agar random
shuffle_idx = np.random.RandomState(42).permutation(len(y_balanced))
X_balanced = X_balanced[shuffle_idx]
y_balanced = y_balanced[shuffle_idx]

print("\n🔀 Dataset telah di-shuffle untuk memastikan randomness")
print(f"✅ Dataset balanced siap digunakan: {len(y_balanced)} sampel")

# Visualisasi perbandingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before balancing
axes[0].bar(urutan_label, [initial_distribution.get(l, 0) for l in urutan_label],
            color=['green', 'orange', 'red'], alpha=0.7)
axes[0].set_title('Distribusi SEBELUM Balancing', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah Sampel', fontsize=12)
axes[0].set_xticklabels(urutan_label, rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# After balancing
axes[1].bar(urutan_label, [final_distribution.get(l, 0) for l in urutan_label],
            color=['green', 'orange', 'red',], alpha=0.7)
axes[1].set_title('Distribusi SETELAH Balancing', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Jumlah Sampel', fontsize=12)
axes[1].set_xticklabels(urutan_label, rotation=45)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
axes[1].axhline(y=target_median, color='blue', linestyle='--',
                label=f'Median Target: {target_median}', linewidth=2)
axes[1].legend()

plt.tight_layout()
plt.show()

# Update variabel X dan y dengan data yang sudah balanced
X = X_balanced
y = y_balanced

In [ ]:
# ====== 1. PERSIAPAN DATA (SETELAH BALANCING) ======
print("="*50)
print("📊 PERSIAPAN DATA - TRAIN TEST SPLIT")
print("="*50)
urutan_label = ['ideal', 'normal', 'bahaya']

# Visualisasi distribusi kelas setelah balancing
plt.figure(figsize=(8, 5))
status_counts = pd.Series(y_balanced).value_counts()
status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
colors = ['green', 'orange', 'red']
bars = plt.bar(urutan_label, status_counts_ordered, color=colors, alpha=0.7, edgecolor='black')

# Tambahkan nilai di atas bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.title('Distribusi Kelas Kualitas Air (Setelah Balancing)', fontsize=14, fontweight='bold')
plt.ylabel('Jumlah', fontsize=12)
plt.xlabel('Status Kualitas Air', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Split data menjadi training dan testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f"\n✅ Data Training: {len(X_train)} sampel ({len(X_train)/len(X_balanced)*100:.1f}%)")
print(f"✅ Data Testing: {len(X_test)} sampel ({len(X_test)/len(X_balanced)*100:.1f}%)")

# Normalisasi data menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("✅ Data berhasil dibagi dan dinormalisasi (StandardScaler)")

# ====== VALIDASI SILANG K-FOLD ======
print("\n" + "="*50)
print("🔍 TUNING HYPERPARAMETER (K-Nearest Neighbors)")
print("="*50)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_values = list(range(3, 16, 2))  # Perluas rentang pencarian
cv_scores = []
cv_stds = []

print("⏳ Mencari nilai K optimal dengan 5-Fold Cross Validation...\n")

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='euclidean')
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    mean_score = scores.mean()
    std_score = scores.std()

    cv_scores.append(mean_score)
    cv_stds.append(std_score)

    print(f"K={k:2d} | Mean CV Accuracy = {mean_score:.4f} | Std = {std_score:.4f}")

best_k = k_values[np.argmax(cv_scores)]
best_acc = max(cv_scores)

print(f"\n🎯 Nilai K Optimal: {best_k}")
print(f"🎯 Best CV Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

# ====== EVALUASI MODEL ======
print("\n" + "="*50)
print("🔍 EVALUASI MODEL (Test Set)")
print("="*50)

best_model = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='euclidean')
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)
weighted_f1 = f1_score(y_test, y_pred, average='weighted')
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n🚀 TRAINING MODEL DENGAN K={best_k}")
print(f"🎯 Test Accuracy      = {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"🎯 Weighted F1-Score  = {weighted_f1:.4f}")
print(f"🎯 Macro F1-Score     = {macro_f1:.4f}")

# Classification Report (dengan urutan label yang benar)
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, labels=urutan_label, target_names=urutan_label, zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
print("\nActual \\ Predicted:", end="")
for label in urutan_label:
    print(f"{label:>12}", end="")
print()
print("-" * 80)
for i, label in enumerate(urutan_label):
    print(f"{label:>18}", end="")
    for j in range(len(urutan_label)):
        print(f"{cm[i][j]:>12}", end="")
    print()

# VISUALISASI 3D - DATA TEST SET MENGGUNAKAN PCA (AMBIL 50 DATA PER KELAS)
print("\n" + "="*50)
print("🎨 VISUALISASI DATA 3D (PCA) - 50 Data per Kelas")
print("="*50)

from sklearn.decomposition import PCA

pca_3d = PCA(n_components=3, random_state=42)
X_test_pca_3d = pca_3d.fit_transform(X_test_scaled)

# Ambil maksimal 50 data per kelas
max_samples_per_class = 50
indices_to_plot = []

print("\n📊 Pengambilan sampel data:")
for label in urutan_label:
    mask = y_test == label
    indices = np.where(mask)[0]

    # Ambil maksimal 50 data (atau semua jika kurang dari 50)
    if len(indices) > max_samples_per_class:
        selected_indices = np.random.choice(indices, max_samples_per_class, replace=False)
    else:
        selected_indices = indices

    indices_to_plot.extend(selected_indices)
    print(f"  • Kelas '{label}': {len(selected_indices)} data dari {len(indices)} total")

indices_to_plot = np.array(indices_to_plot)

# Buat visualisasi 3D dengan data terpilih
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

colors_map = {'ideal': 'green', 'normal': 'orange', 'bahaya': 'red'}
markers_map = {'ideal': 'o', 'normal': 's', 'bahaya': '^'}

# Plot setiap kelas (hanya data terpilih)
for label in urutan_label:
    mask = y_test == label
    mask_selected = np.isin(np.arange(len(y_test)), indices_to_plot) & mask

    ax.scatter(X_test_pca_3d[mask_selected, 0],
               X_test_pca_3d[mask_selected, 1],
               X_test_pca_3d[mask_selected, 2],
               c=colors_map[label],
               marker=markers_map[label],
               label=f'Aktual: {label}',
               alpha=0.7,
               s=100,
               edgecolors='black',
               linewidth=0.5)

# Tambahkan marker untuk prediksi yang salah (dari data terpilih)
misclassified = (y_test != y_pred) & np.isin(np.arange(len(y_test)), indices_to_plot)
if misclassified.sum() > 0:
    ax.scatter(X_test_pca_3d[misclassified, 0],
               X_test_pca_3d[misclassified, 1],
               X_test_pca_3d[misclassified, 2],
               marker='x', s=300, c='black', linewidth=3,
               label=f'Salah Klasifikasi ({misclassified.sum()})',
               zorder=5)

ax.set_xlabel(f'\nPC1 ({pca_3d.explained_variance_ratio_[0]*100:.1f}%)',
              fontsize=11, fontweight='bold')
ax.set_ylabel(f'\nPC2 ({pca_3d.explained_variance_ratio_[1]*100:.1f}%)',
              fontsize=11, fontweight='bold')
ax.set_zlabel(f'\nPC3 ({pca_3d.explained_variance_ratio_[2]*100:.1f}%)',
              fontsize=11, fontweight='bold')
ax.set_title(f'Visualisasi 3D Data Test Set (PCA)\n[Menampilkan {len(indices_to_plot)} dari {len(X_test)} total data]',
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=10, framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)

# Rotate untuk sudut pandang terbaik
ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.show()

print(f"\n✅ Variance explained by 3 components: {sum(pca_3d.explained_variance_ratio_)*100:.2f}%")
print(f"✅ Total data yang divisualisasikan: {len(indices_to_plot)} data")

# ====== ANALISIS LEARNING CURVE ======
print("\n" + "="*50)
print("📈 ANALISIS LEARNING CURVE")
print("="*50)

train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', color='blue', linewidth=2,
         markersize=8, label='Training Score')
plt.plot(train_sizes, test_mean, 'o-', color='green', linewidth=2,
         markersize=8, label='Cross-validation Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.15, color='blue')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std,
                 alpha=0.15, color='green')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Training Examples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - KNN Classifier (Data Balanced)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

# Analisis hasil learning curve
print("\n🔍 Analisis Learning Curve:")
gap = train_mean[-1] - test_mean[-1]
print(f"  - Training Score (final): {train_mean[-1]:.4f}")
print(f"  - CV Score (final):       {test_mean[-1]:.4f}")
print(f"  - Gap:                    {gap:.4f}")

if test_mean[-1] > 0.9 and train_mean[-1] > 0.9 and gap < 0.05:
    performance = "✅ Model terlihat SANGAT BAIK (high performance, low overfitting)"
elif gap > 0.1:
    performance = "⚠️ Model mungkin OVERFITTING (training score >> CV score)"
elif train_mean[-1] < 0.8:
    performance = "⚠️ Model mungkin UNDERFITTING (perlu model lebih kompleks)"
elif test_mean[-1] > 0.85:
    performance = "✅ Model memiliki performa BAIK dan SEIMBANG"
else:
    performance = "⚠️ Model memiliki performa CUKUP (bisa ditingkatkan)"

print(f"  Kesimpulan: {performance}")

In [ ]:
# 📁 Tentukan folder penyimpanan
output_dir = '/content/drive/MyDrive/hasil_machine_learning/3-Rule-Based-priority/'
os.makedirs(output_dir, exist_ok=True)

# ====== SIMPAN SEMUA OUTPUT ======
print("\n" + "="*60)
print("💾 PENYIMPANAN HASIL KE GOOGLE DRIVE")
print("="*60)

# 1. Simpan Model KNN
model_path = os.path.join(output_dir, 'knn_water_quality_model.pkl')
joblib.dump(best_model, model_path)
print(f"✅ Model KNN disimpan: {model_path}")

# 2. Simpan Scaler
scaler_path = os.path.join(output_dir, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"✅ Scaler disimpan: {scaler_path}")

# 3. Simpan Feature Columns
features_path = os.path.join(output_dir, 'feature_columns.pkl')
joblib.dump(feature_columns, features_path)
print(f"✅ Feature columns disimpan: {features_path}")

# 4. Simpan Dataset dengan Status
dataset_path = os.path.join(output_dir, 'dataset_balanced_with_status.csv')
df_balanced = pd.DataFrame(X_balanced, columns=feature_columns)
numeric_columns = ['pH(ph_units)', 'Temp(cel)', 'TDS(ppm)', 'DO(mg/l)']
for col in numeric_columns:
    if col in df_balanced.columns:
        df_balanced[col] = df_balanced[col].astype(float).round(2)
df_balanced['status'] = y_balanced
df_balanced.to_csv(dataset_path, index=False)
print(f"✅ Dataset balanced disimpan: {dataset_path}")

# 5. Simpan Dataset Original dengan Status
dataset_original_path = os.path.join(output_dir, 'dataset_original_with_status.csv')
df.to_csv(dataset_original_path, index=False)
print(f"✅ Dataset original disimpan: {dataset_original_path}")

# 6. Simpan Metadata Model
metadata_path = os.path.join(output_dir, 'model_metadata.txt')
with open(metadata_path, 'w', encoding='utf-8') as f:
    f.write("="*60 + "\n")
    f.write("MODEL METADATA - KNN WATER QUALITY CLASSIFIER\n")
    f.write("="*60 + "\n\n")

    f.write("📊 INFORMASI DATASET\n")
    f.write("-" * 60 + "\n")
    f.write(f"Total Data (Original):        {len(df)} sampel\n")
    f.write(f"Total Data (After Balancing): {len(y_balanced)} sampel\n")
    f.write(f"Training Set:                 {len(y_train)} sampel ({len(y_train)/len(y_balanced)*100:.1f}%)\n")
    f.write(f"Testing Set:                  {len(y_test)} sampel ({len(y_test)/len(y_balanced)*100:.1f}%)\n")
    f.write(f"Jumlah Fitur:                 {len(feature_columns)}\n")
    f.write(f"Nama Fitur:                   {', '.join(feature_columns)}\n\n")

    f.write("🎯 DISTRIBUSI KELAS (BEFORE BALANCING)\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    status_counts = df['status'].value_counts()
    status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
    for label in urutan_label:
        count = status_counts_ordered[label]
        f.write(f"  {label:12s}: {count:5d} ({count/len(df)*100:5.2f}%)\n")
    f.write("\n")

    f.write("🎯 DISTRIBUSI KELAS (AFTER BALANCING)\n")
    f.write("-" * 60 + "\n")
    for label in urutan_label:
        count = np.sum(y_balanced == label)
        f.write(f"  {label:12s}: {count:5d} sampel ({count/len(y_balanced)*100:5.2f}%)\n")
    f.write("\n")

    f.write("🔍 PARAMETER MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Algoritma:                    K-Nearest Neighbors (KNN)\n")
    f.write(f"Nilai K Optimal:              {best_k}\n")
    f.write(f"Weights:                      distance\n")
    f.write(f"Metric:                       euclidean\n")
    f.write(f"Normalisasi:                  StandardScaler\n\n")

    f.write("📈 PERFORMA MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Best CV Accuracy:             {best_acc:.4f} ({best_acc*100:.2f}%)\n")
    f.write(f"Test Accuracy:                {test_accuracy:.4f} ({test_accuracy*100:.2f}%)\n")
    f.write(f"Weighted F1-Score:            {weighted_f1:.4f}\n")
    f.write(f"Macro F1-Score:               {macro_f1:.4f}\n\n")

    f.write("📊 AKURASI PER KELAS\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
    for i, label in enumerate(urutan_label):
        class_correct = cm[i, i]
        class_total = cm[i, :].sum()
        class_accuracy = class_correct / class_total if class_total > 0 else 0
        f.write(f"  {label:12s}: {class_accuracy:.4f} ({class_accuracy*100:.2f}%) - {class_correct}/{class_total} benar\n")
    f.write("\n")

    f.write("📋 CLASSIFICATION REPORT\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    f.write(classification_report(y_test, y_pred, labels=urutan_label, target_names=urutan_label, zero_division=0))
    f.write("\n")

    f.write("🔬 CONFUSION MATRIX\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    f.write("                              Predicted\n")
    f.write("Actual        " + "  ".join([f"{l:8s}" for l in urutan_label]) + "\n")
    for i, label in enumerate(urutan_label):
        f.write(f"{label:9s} " + "  ".join([f"{cm[i,j]:8d}" for j in range(len(urutan_label))]) + "\n")
    f.write("\n")

print(f"✅ Metadata model disimpan: {metadata_path}")

# Simpan Visualisasi
# Visualisasi 1: Distribusi Kelas Balanced
plt.figure(figsize=(8, 5))
status_counts = pd.Series(y_balanced).value_counts()
status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
colors = ['green', 'orange', 'red']
bars = plt.bar(urutan_label, status_counts_ordered, color=colors, alpha=0.7, edgecolor='black')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.title('Distribusi Kelas Kualitas Air (Setelah Balancing)', fontsize=14, fontweight='bold')
plt.ylabel('Jumlah', fontsize=12)
plt.xlabel('Status Kualitas Air', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'class_distribution_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik distribusi disimpan")

# Visualisasi 2: KNN Cross-Validation Results
plt.figure(figsize=(10, 6))
plt.errorbar(k_values, cv_scores, yerr=cv_stds, fmt='o-', capsize=5,
             color='blue', ecolor='red', linewidth=2, markersize=8)
plt.axhline(y=max(cv_scores), color='green', linestyle='--', alpha=0.7, linewidth=2)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Nilai K', fontsize=12)
plt.ylabel('Accuracy (5-Fold CV)', fontsize=12)
plt.title('Performa KNN terhadap Nilai K (Data Balanced)', fontsize=14, fontweight='bold')
plt.xticks(k_values)
plt.scatter(best_k, best_acc, color='red', s=200, zorder=5,
            label=f'Best K={best_k} (Acc={best_acc:.4f})', marker='*')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'knn_cv_results_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik CV results disimpan")

# Visualisasi 3: Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=urutan_label, yticklabels=urutan_label,
            annot_kws={"size": 14, "weight": "bold"},
            cbar_kws={'label': 'Jumlah Sampel'},
            linewidths=1, linecolor='gray')
plt.xlabel('Predicted', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=14, fontweight='bold')
plt.title('Confusion Matrix (Test Set - Data Balanced)', fontsize=16, fontweight='bold')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'confusion_matrix_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik confusion matrix disimpan")

# Visualisasi 4: Learning Curve
train_sizes_lc, train_scores_lc, test_scores_lc = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)
train_mean_lc = np.mean(train_scores_lc, axis=1)
train_std_lc = np.std(train_scores_lc, axis=1)
test_mean_lc = np.mean(test_scores_lc, axis=1)
test_std_lc = np.std(test_scores_lc, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes_lc, train_mean_lc, 'o-', color='blue', linewidth=2,
         markersize=8, label='Training Score')
plt.plot(train_sizes_lc, test_mean_lc, 'o-', color='green', linewidth=2,
         markersize=8, label='Cross-validation Score')
plt.fill_between(train_sizes_lc, train_mean_lc - train_std_lc, train_mean_lc + train_std_lc,
                 alpha=0.15, color='blue')
plt.fill_between(train_sizes_lc, test_mean_lc - test_std_lc, test_mean_lc + test_std_lc,
                 alpha=0.15, color='green')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Training Examples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - KNN Classifier (Data Balanced)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'learning_curve_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik learning curve disimpan")


# Ringkasan File yang Tersimpan
print("\n" + "="*60)
print("📁 DAFTAR FILE TERSIMPAN")
print("="*60)
saved_files = [
    ('Model KNN', 'knn_water_quality_model.pkl'),
    ('Scaler', 'scaler.pkl'),
    ('Feature Columns', 'feature_columns.pkl'),
    ('Dataset Balanced', 'dataset_balanced_with_status.csv'),
    ('Dataset Original', 'dataset_original_with_status.csv'),
    ('Metadata', 'model_metadata.txt'),
    ('Grafik Distribusi', 'class_distribution_balanced.png'),
    ('Grafik CV Results', 'knn_cv_results_balanced.png'),
    ('Grafik Confusion Matrix', 'confusion_matrix_balanced.png'),
    ('Grafik Learning Curve', 'learning_curve_balanced.png')
]

for name, filename in saved_files:
    filepath = os.path.join(output_dir, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        size_str = f"{size/1024:.2f} KB" if size < 1024*1024 else f"{size/(1024*1024):.2f} MB"
        print(f"✅ {name:25s}: {filename:40s} ({size_str})")
    else:
        print(f"❌ {name:25s}: {filename:40s} (TIDAK DITEMUKAN)")

## **METODE KETIGA:** Z-score Normalization

In [ ]:
# Threshold Dinamis Berbasis Statistik (Z-Score Normalization)
print("\nStatistical Summary:")
print(df.describe())

def calculate_zscore_thresholds(data, column_name):
    mean = data[column_name].mean()
    std = data[column_name].std()

    # Hindari std = 0 (jika semua nilai sama)
    if std == 0:
        std = 1e-6  # Nilai kecil agar tidak error

    thresholds = {
        'mean': mean,
        'std': std,
        'ideal_lower': mean - 1.0 * std,
        'ideal_upper': mean + 1.0 * std,
        'normal_lower': mean - 2.0 * std,
        'normal_upper': mean + 2.0 * std,
    }

    print(f"\n{column_name} - Threshold Dinamis (Z-Score):")
    print(f"  Mean: {mean:.2f}, Std: {std:.2f}")
    print(f"  Ideal: {thresholds['ideal_lower']:.2f} - {thresholds['ideal_upper']:.2f}")
    print(f"  Normal: {thresholds['normal_lower']:.2f} - {thresholds['ideal_lower']:.2f} dan {thresholds['ideal_upper']:.2f} - {thresholds['normal_upper']:.2f}")
    print(f"  Bahaya: < {thresholds['normal_lower']:.2f} atau > {thresholds['normal_upper']:.2f}")

    return thresholds

def classify_by_zscore(value, thresholds):
    if pd.isna(value):
        return 'normal'
    # Cek Ideal Dulu
    if thresholds['ideal_lower'] <= value <= thresholds['ideal_upper']:
        return 'ideal'
    # Cek Normal
    elif thresholds['normal_lower'] <= value <= thresholds['normal_upper']:
        return 'normal'
    # sisa cek bahaya
    else:
        return 'bahaya'

# Mapping kolom yang mungkin ada di dataset
column_mapping = {
    'pH': ['pH', 'water_pH', 'pH(ph_units)'],
    'temperature': ['temperature', 'Temp(cel)', 'water_temp'],
    'TDS': ['TDS(ppm)', 'TDS'],
    'DO': ['do', 'DO', 'DO(mg/l)']
}

# Cari kolom yang tersedia di dataset
available_params = {}
for param, possible_names in column_mapping.items():
    for col_name in possible_names:
        if col_name in df.columns:
            available_params[param] = col_name
            break

print("\n" + "="*80)
print("PARAMETER YANG DITEMUKAN:")
print("="*80)
for param, col in available_params.items():
    print(f"  {param}: {col}")

# Hitung threshold dinamis untuk setiap parameter
thresholds_dict = {}
for param, col_name in available_params.items():
    thresholds_dict[param] = calculate_zscore_thresholds(df, col_name)

# Fungsi klasifikasi dengan threshold dinamis
def classify_water_quality_dynamic(row):
    prioritas_order = ['bahaya', 'normal', 'ideal']
    kondisi_terbaik = 'ideal'

    # Klasifikasi setiap parameter yang tersedia
    for param, col_name in available_params.items():
        if col_name in row:
            value = row[col_name]
            if pd.notna(value):
                kondisi = classify_by_zscore(value, thresholds_dict[param])

                # Update kondisi terbaik (prioritas terburuk)
                if prioritas_order.index(kondisi) < prioritas_order.index(kondisi_terbaik):
                    kondisi_terbaik = kondisi

    return kondisi_terbaik

# Tambahkan kolom status dengan metode dinamis
df['status'] = df.apply(classify_water_quality_dynamic, axis=1)

# Definisikan urutan dari TERBAIK ke TERBURUK
order_from_best_to_worst = ['ideal', 'normal', 'bahaya']
status_counts = df['status'].value_counts()
status_counts_ordered = status_counts.reindex(order_from_best_to_worst, fill_value=0)
print("\nDistribusi Status (Terbaik → Terburuk):")
print(status_counts_ordered)

# Pilih fitur untuk training (sesuaikan dengan kolom di dataset)
feature_columns = [col for col in df.columns if col not in ['id','created_date','entry_id','created_at','status']]

In [ ]:
# ====== BALANCING DATASET SECARA DINAMIS BERBASIS MEDIAN ======
print("\n" + "="*60)
print("⚖️  MENYEIMBANGKAN DATASET (DINAMIS - BERDASARKAN MEDIAN)")
print("="*60)

X = df[feature_columns].values
y = df['status'].values
urutan_label = ['ideal', 'normal', 'bahaya']

# Tampilkan distribusi awal
print("\n📊 Distribusi Kelas SEBELUM Balancing:")
initial_distribution = {}
for label in urutan_label:
    initial_distribution[label] = np.sum(y == label)

for label in urutan_label:
    count = initial_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y):5d} sampel")

# Hitung median sebagai target jumlah sampel per kelas
jumlah_per_kelas = [initial_distribution[label] for label in urutan_label]
target_median = int(np.median(jumlah_per_kelas))

print(f"\n🎯 Median jumlah sampel per kelas: {target_median}")
print("📊 Strategi Dinamis:")
print("   - Kelas < median → oversample")
print("   - Kelas > median → undersample")
print("   - Kelas = median → pertahankan")

# Buat strategi sampling
over_strategy = {}
under_strategy = {}

for label in urutan_label:
    count = initial_distribution[label]
    if count < target_median:
        over_strategy[label] = target_median
    elif count > target_median:
        under_strategy[label] = target_median
    # Jika sama, tidak perlu sampling

# Bangun pipeline hanya jika diperlukan
steps = []

# Tambahkan oversampling jika ada kelas minority
if over_strategy:
    steps.append(('oversample', SMOTE(
        sampling_strategy=over_strategy,
        random_state=42,
        k_neighbors=min(3, min(initial_distribution.values()))  # hindari error jika sampel < 3
    )))

# Tambahkan undersampling jika ada kelas majority
if under_strategy:
    steps.append(('undersample', RandomUnderSampler(
        sampling_strategy=under_strategy,
        random_state=42
    )))

if steps:
    pipeline = Pipeline(steps)
    print("\n⏳ Proses balancing dinamis...")
    X_balanced, y_balanced = pipeline.fit_resample(X, y)
else:
    print("\n⚠️  Tidak diperlukan balancing (semua kelas sudah seimbang).")
    X_balanced, y_balanced = X.copy(), y.copy()

# Tampilkan distribusi setelah balancing
print("\n✅ Distribusi Kelas SETELAH Balancing:")
final_distribution = {}
for label in urutan_label:
    final_distribution[label] = np.sum(y_balanced == label)

for label in urutan_label:
    count = final_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y_balanced)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y_balanced):5d} sampel")

# Shuffle dataset agar random
shuffle_idx = np.random.RandomState(42).permutation(len(y_balanced))
X_balanced = X_balanced[shuffle_idx]
y_balanced = y_balanced[shuffle_idx]

print("\n🔀 Dataset telah di-shuffle untuk memastikan randomness")
print(f"✅ Dataset balanced siap digunakan: {len(y_balanced)} sampel")

# Visualisasi perbandingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before balancing
axes[0].bar(urutan_label, [initial_distribution.get(l, 0) for l in urutan_label],
            color=['green', 'orange', 'red'], alpha=0.7)
axes[0].set_title('Distribusi SEBELUM Balancing', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah Sampel', fontsize=12)
axes[0].set_xticklabels(urutan_label, rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# After balancing
axes[1].bar(urutan_label, [final_distribution.get(l, 0) for l in urutan_label],
            color=['green', 'orange', 'red'], alpha=0.7)
axes[1].set_title('Distribusi SETELAH Balancing', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Jumlah Sampel', fontsize=12)
axes[1].set_xticklabels(urutan_label, rotation=45)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
axes[1].axhline(y=target_median, color='blue', linestyle='--',
                label=f'Median Target: {target_median}', linewidth=2)
axes[1].legend()

plt.tight_layout()
plt.show()

# Update variabel X dan y dengan data yang sudah balanced
X = X_balanced
y = y_balanced

In [ ]:
# ====== 1. PERSIAPAN DATA (SETELAH BALANCING) ======
print("="*50)
print("📊 PERSIAPAN DATA - TRAIN TEST SPLIT")
print("="*50)
urutan_label = ['ideal', 'normal', 'bahaya']

# Visualisasi distribusi kelas setelah balancing
plt.figure(figsize=(8, 5))
status_counts = pd.Series(y_balanced).value_counts()
status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
colors = ['green', 'orange', 'red']
bars = plt.bar(urutan_label, status_counts_ordered, color=colors, alpha=0.7, edgecolor='black')

# Tambahkan nilai di atas bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.title('Distribusi Kelas Kualitas Air (Setelah Balancing)', fontsize=14, fontweight='bold')
plt.ylabel('Jumlah', fontsize=12)
plt.xlabel('Status Kualitas Air', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Split data menjadi training dan testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f"\n✅ Data Training: {len(X_train)} sampel ({len(X_train)/len(X_balanced)*100:.1f}%)")
print(f"✅ Data Testing: {len(X_test)} sampel ({len(X_test)/len(X_balanced)*100:.1f}%)")

# Normalisasi data menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("✅ Data berhasil dibagi dan dinormalisasi (StandardScaler)")

# ====== VALIDASI SILANG K-FOLD ======
print("\n" + "="*50)
print("🔍 TUNING HYPERPARAMETER (K-Nearest Neighbors)")
print("="*50)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_values = list(range(3, 16, 2))  # Perluas rentang pencarian
cv_scores = []
cv_stds = []

print("⏳ Mencari nilai K optimal dengan 5-Fold Cross Validation...\n")

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='euclidean')
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    mean_score = scores.mean()
    std_score = scores.std()

    cv_scores.append(mean_score)
    cv_stds.append(std_score)

    print(f"K={k:2d} | Mean CV Accuracy = {mean_score:.4f} | Std = {std_score:.4f}")

best_k = k_values[np.argmax(cv_scores)]
best_acc = max(cv_scores)

print(f"\n🎯 Nilai K Optimal: {best_k}")
print(f"🎯 Best CV Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

# ====== EVALUASI MODEL ======
print("\n" + "="*50)
print("🔍 EVALUASI MODEL (Test Set)")
print("="*50)

best_model = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='euclidean')
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)
weighted_f1 = f1_score(y_test, y_pred, average='weighted')
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n🚀 TRAINING MODEL DENGAN K={best_k}")
print(f"🎯 Test Accuracy      = {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"🎯 Weighted F1-Score  = {weighted_f1:.4f}")
print(f"🎯 Macro F1-Score     = {macro_f1:.4f}")

# Classification Report (dengan urutan label yang benar)
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, labels=urutan_label, target_names=urutan_label, zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
print("\nActual \\ Predicted:", end="")
for label in urutan_label:
    print(f"{label:>12}", end="")
print()
print("-" * 80)
for i, label in enumerate(urutan_label):
    print(f"{label:>18}", end="")
    for j in range(len(urutan_label)):
        print(f"{cm[i][j]:>12}", end="")
    print()

# ====== ANALISIS LEARNING CURVE ======
print("\n" + "="*50)
print("📈 ANALISIS LEARNING CURVE")
print("="*50)

train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', color='blue', linewidth=2,
         markersize=8, label='Training Score')
plt.plot(train_sizes, test_mean, 'o-', color='green', linewidth=2,
         markersize=8, label='Cross-validation Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.15, color='blue')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std,
                 alpha=0.15, color='green')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Training Examples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - KNN Classifier (Data Balanced)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

# Analisis hasil learning curve
print("\n🔍 Analisis Learning Curve:")
gap = train_mean[-1] - test_mean[-1]
print(f"  - Training Score (final): {train_mean[-1]:.4f}")
print(f"  - CV Score (final):       {test_mean[-1]:.4f}")
print(f"  - Gap:                    {gap:.4f}")

if test_mean[-1] > 0.9 and train_mean[-1] > 0.9 and gap < 0.05:
    performance = "✅ Model terlihat SANGAT BAIK (high performance, low overfitting)"
elif gap > 0.1:
    performance = "⚠️ Model mungkin OVERFITTING (training score >> CV score)"
elif train_mean[-1] < 0.8:
    performance = "⚠️ Model mungkin UNDERFITTING (perlu model lebih kompleks)"
elif test_mean[-1] > 0.85:
    performance = "✅ Model memiliki performa BAIK dan SEIMBANG"
else:
    performance = "⚠️ Model memiliki performa CUKUP (bisa ditingkatkan)"

print(f"  Kesimpulan: {performance}")

In [ ]:
# 📁 Tentukan folder penyimpanan
output_dir = '/content/drive/MyDrive/hasil_machine_learning/3-z-score-normalization/'
os.makedirs(output_dir, exist_ok=True)

# ====== SIMPAN SEMUA OUTPUT ======
print("\n" + "="*60)
print("💾 PENYIMPANAN HASIL KE GOOGLE DRIVE")
print("="*60)

# 1. Simpan Model KNN
model_path = os.path.join(output_dir, 'knn_water_quality_model.pkl')
joblib.dump(best_model, model_path)
print(f"✅ Model KNN disimpan: {model_path}")

# 2. Simpan Scaler
scaler_path = os.path.join(output_dir, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"✅ Scaler disimpan: {scaler_path}")

# 3. Simpan Feature Columns
features_path = os.path.join(output_dir, 'feature_columns.pkl')
joblib.dump(feature_columns, features_path)
print(f"✅ Feature columns disimpan: {features_path}")

# 4. Simpan Dataset dengan Status
dataset_path = os.path.join(output_dir, 'dataset_balanced_with_status.csv')
df_balanced = pd.DataFrame(X_balanced, columns=feature_columns)
numeric_columns = ['pH(ph_units)', 'Temp(cel)', 'TDS(ppm)', 'DO(mg/l)']
for col in numeric_columns:
    if col in df_balanced.columns:
        df_balanced[col] = df_balanced[col].astype(float).round(2)
df_balanced['status'] = y_balanced
df_balanced.to_csv(dataset_path, index=False)
print(f"✅ Dataset balanced disimpan: {dataset_path}")

# 5. Simpan Dataset Original dengan Status
dataset_original_path = os.path.join(output_dir, 'dataset_original_with_status.csv')
df.to_csv(dataset_original_path, index=False)
print(f"✅ Dataset original disimpan: {dataset_original_path}")

# 6. Simpan Metadata Model
metadata_path = os.path.join(output_dir, 'model_metadata.txt')
with open(metadata_path, 'w', encoding='utf-8') as f:
    f.write("="*60 + "\n")
    f.write("MODEL METADATA - KNN WATER QUALITY CLASSIFIER\n")
    f.write("="*60 + "\n\n")

    f.write("📊 INFORMASI DATASET\n")
    f.write("-" * 60 + "\n")
    f.write(f"Total Data (Original):        {len(df)} sampel\n")
    f.write(f"Total Data (After Balancing): {len(y_balanced)} sampel\n")
    f.write(f"Training Set:                 {len(y_train)} sampel ({len(y_train)/len(y_balanced)*100:.1f}%)\n")
    f.write(f"Testing Set:                  {len(y_test)} sampel ({len(y_test)/len(y_balanced)*100:.1f}%)\n")
    f.write(f"Jumlah Fitur:                 {len(feature_columns)}\n")
    f.write(f"Nama Fitur:                   {', '.join(feature_columns)}\n\n")

    f.write("🎯 DISTRIBUSI KELAS (BEFORE BALANCING)\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    status_counts = df['status'].value_counts()
    status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
    for label in urutan_label:
        count = status_counts_ordered[label]
        f.write(f"  {label:12s}: {count:5d} ({count/len(df)*100:5.2f}%)\n")
    f.write("\n")

    f.write("🎯 DISTRIBUSI KELAS (AFTER BALANCING)\n")
    f.write("-" * 60 + "\n")
    for label in urutan_label:
        count = np.sum(y_balanced == label)
        f.write(f"  {label:12s}: {count:5d} ({count/len(y_balanced)*100:5.2f}%)\n")
    f.write("\n")

    f.write("🔍 PARAMETER MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Algoritma:                    K-Nearest Neighbors (KNN)\n")
    f.write(f"Nilai K Optimal:              {best_k}\n")
    f.write(f"Weights:                      distance\n")
    f.write(f"Metric:                       euclidean\n")
    f.write(f"Normalisasi:                  StandardScaler\n\n")

    f.write("📈 PERFORMA MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Best CV Accuracy:             {best_acc:.4f} ({best_acc*100:.2f}%)\n")
    f.write(f"Test Accuracy:                {test_accuracy:.4f} ({test_accuracy*100:.2f}%)\n")
    f.write(f"Weighted F1-Score:            {weighted_f1:.4f}\n")
    f.write(f"Macro F1-Score:               {macro_f1:.4f}\n\n")

    f.write("📊 AKURASI PER KELAS\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
    for i, label in enumerate(urutan_label):
        class_correct = cm[i, i]
        class_total = cm[i, :].sum()
        class_accuracy = class_correct / class_total if class_total > 0 else 0
        f.write(f"  {label:12s}: {class_accuracy:.4f} ({class_accuracy*100:.2f}%) - {class_correct}/{class_total} benar\n")
    f.write("\n")

    f.write("📋 CLASSIFICATION REPORT\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    f.write(classification_report(y_test, y_pred, labels=urutan_label, target_names=urutan_label, zero_division=0))
    f.write("\n")

    f.write("🔬 CONFUSION MATRIX\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    f.write("                              Predicted\n")
    f.write("Actual        " + "  ".join([f"{l:8s}" for l in urutan_label]) + "\n")
    for i, label in enumerate(urutan_label):
        f.write(f"{label:9s} " + "  ".join([f"{cm[i,j]:8d}" for j in range(len(urutan_label))]) + "\n")
    f.write("\n")

print(f"✅ Metadata model disimpan: {metadata_path}")

# Simpan Visualisasi
# Visualisasi 1: Distribusi Kelas Balanced
plt.figure(figsize=(8, 5))
status_counts = pd.Series(y_balanced).value_counts()
status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
colors = ['green', 'orange', 'red']
bars = plt.bar(urutan_label, status_counts_ordered, color=colors, alpha=0.7, edgecolor='black')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.title('Distribusi Kelas Kualitas Air (Setelah Balancing)', fontsize=14, fontweight='bold')
plt.ylabel('Jumlah', fontsize=12)
plt.xlabel('Status Kualitas Air', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'class_distribution_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik distribusi disimpan")

# Visualisasi 2: KNN Cross-Validation Results
plt.figure(figsize=(10, 6))
plt.errorbar(k_values, cv_scores, yerr=cv_stds, fmt='o-', capsize=5,
             color='blue', ecolor='red', linewidth=2, markersize=8)
plt.axhline(y=max(cv_scores), color='green', linestyle='--', alpha=0.7, linewidth=2)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Nilai K', fontsize=12)
plt.ylabel('Accuracy (5-Fold CV)', fontsize=12)
plt.title('Performa KNN terhadap Nilai K (Data Balanced)', fontsize=14, fontweight='bold')
plt.xticks(k_values)
plt.scatter(best_k, best_acc, color='red', s=200, zorder=5,
            label=f'Best K={best_k} (Acc={best_acc:.4f})', marker='*')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'knn_cv_results_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik CV results disimpan")

# Visualisasi 3: Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=urutan_label, yticklabels=urutan_label,
            annot_kws={"size": 14, "weight": "bold"},
            cbar_kws={'label': 'Jumlah Sampel'},
            linewidths=1, linecolor='gray')
plt.xlabel('Predicted', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=14, fontweight='bold')
plt.title('Confusion Matrix (Test Set - Data Balanced)', fontsize=16, fontweight='bold')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'confusion_matrix_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik confusion matrix disimpan")

# Visualisasi 4: Learning Curve
train_sizes_lc, train_scores_lc, test_scores_lc = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)
train_mean_lc = np.mean(train_scores_lc, axis=1)
train_std_lc = np.std(train_scores_lc, axis=1)
test_mean_lc = np.mean(test_scores_lc, axis=1)
test_std_lc = np.std(test_scores_lc, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes_lc, train_mean_lc, 'o-', color='blue', linewidth=2,
         markersize=8, label='Training Score')
plt.plot(train_sizes_lc, test_mean_lc, 'o-', color='green', linewidth=2,
         markersize=8, label='Cross-validation Score')
plt.fill_between(train_sizes_lc, train_mean_lc - train_std_lc, train_mean_lc + train_std_lc,
                 alpha=0.15, color='blue')
plt.fill_between(train_sizes_lc, test_mean_lc - test_std_lc, test_mean_lc + test_std_lc,
                 alpha=0.15, color='green')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Training Examples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - KNN Classifier (Data Balanced)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'learning_curve_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik learning curve disimpan")


# Ringkasan File yang Tersimpan
print("\n" + "="*60)
print("📁 DAFTAR FILE TERSIMPAN")
print("="*60)
saved_files = [
    ('Model KNN', 'knn_water_quality_model.pkl'),
    ('Scaler', 'scaler.pkl'),
    ('Feature Columns', 'feature_columns.pkl'),
    ('Dataset Balanced', 'dataset_balanced_with_status.csv'),
    ('Dataset Original', 'dataset_original_with_status.csv'),
    ('Metadata', 'model_metadata.txt'),
    ('Grafik Distribusi', 'class_distribution_balanced.png'),
    ('Grafik CV Results', 'knn_cv_results_balanced.png'),
    ('Grafik Confusion Matrix', 'confusion_matrix_balanced.png'),
    ('Grafik Learning Curve', 'learning_curve_balanced.png')
]

for name, filename in saved_files:
    filepath = os.path.join(output_dir, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        size_str = f"{size/1024:.2f} KB" if size < 1024*1024 else f"{size/(1024*1024):.2f} MB"
        print(f"✅ {name:25s}: {filename:40s} ({size_str})")
    else:
        print(f"❌ {name:25s}: {filename:40s} (TIDAK DITEMUKAN)")

## **METODE KEEMPAT:** Fuzzy Membership Functions

In [ ]:
# FUZZY MEMBERSHIP FUNCTIONS
class FuzzyWaterQuality:
    """
    Kelas untuk mendefinisikan fungsi keanggotaan fuzzy
    dengan 3 status: ideal, normal, bahaya
    """
    def __init__(self):
        # Definisi range untuk setiap parameter
        self.ph_range = np.arange(0, 14.1, 0.1)
        self.temp_range = np.arange(0, 50.1, 0.1)
        self.tds_range = np.arange(0, 2000.1, 1)
        self.do_range = np.arange(0, 20.1, 0.1)

        # Setup fuzzy membership functions
        self._setup_ph_fuzzy()
        self._setup_temp_fuzzy()
        self._setup_tds_fuzzy()
        self._setup_do_fuzzy()

    def _setup_ph_fuzzy(self):
        """Fungsi keanggotaan fuzzy untuk pH"""
        self.ph_bahaya_low = fuzz.trapmf(self.ph_range, [0, 0, 5.5, 6.0])
        self.ph_bahaya_high = fuzz.trapmf(self.ph_range, [8.0, 8.5, 14, 14])
        self.ph_normal_low = fuzz.trimf(self.ph_range, [5.8, 6.3, 6.7])
        self.ph_normal_high = fuzz.trimf(self.ph_range, [7.3, 7.7, 8.2])
        self.ph_ideal = fuzz.trimf(self.ph_range, [6.7, 7.0, 7.3])

    def _setup_temp_fuzzy(self):
        """Fungsi keanggotaan fuzzy untuk Suhu"""
        self.temp_bahaya_low = fuzz.trapmf(self.temp_range, [0, 0, 18, 20])
        self.temp_bahaya_high = fuzz.trapmf(self.temp_range, [34, 36, 50, 50])
        self.temp_normal_low = fuzz.trimf(self.temp_range, [20, 21.5, 23])
        self.temp_normal_high = fuzz.trimf(self.temp_range, [31, 32.5, 34])
        self.temp_ideal = fuzz.trimf(self.temp_range, [24, 27, 30])

    def _setup_tds_fuzzy(self):
        """Fungsi keanggotaan fuzzy untuk TDS"""
        self.tds_ideal = fuzz.trimf(self.tds_range, [100, 200, 350])
        self.tds_normal = fuzz.trimf(self.tds_range, [300, 500, 750])
        self.tds_bahaya = fuzz.trapmf(self.tds_range, [700, 1000, 2000, 2000])

    def _setup_do_fuzzy(self):
        """Fungsi keanggotaan fuzzy untuk DO """
        self.do_bahaya = fuzz.trapmf(self.do_range, [0, 0, 2.0, 3.0])
        self.do_normal = fuzz.trimf(self.do_range, [3.0, 4.5, 5.5])
        self.do_ideal = fuzz.trapmf(self.do_range, [5.0, 6.0, 20, 20])

    def get_membership_values(self, ph, temp, tds, do):
        """Menghitung nilai keanggotaan untuk semua kategori"""
        ph = np.clip(ph, self.ph_range[0], self.ph_range[-1])
        temp = np.clip(temp, self.temp_range[0], self.temp_range[-1])
        tds = np.clip(tds, self.tds_range[0], self.tds_range[-1])
        do = np.clip(do, self.do_range[0], self.do_range[-1])

        memberships = {
            # pH
            'ph_ideal': fuzz.interp_membership(self.ph_range, self.ph_ideal, ph),
            'ph_normal_low': fuzz.interp_membership(self.ph_range, self.ph_normal_low, ph),
            'ph_normal_high': fuzz.interp_membership(self.ph_range, self.ph_normal_high, ph),
            'ph_bahaya_low': fuzz.interp_membership(self.ph_range, self.ph_bahaya_low, ph),
            'ph_bahaya_high': fuzz.interp_membership(self.ph_range, self.ph_bahaya_high, ph),

            # Temperature
            'temp_ideal': fuzz.interp_membership(self.temp_range, self.temp_ideal, temp),
            'temp_normal_low': fuzz.interp_membership(self.temp_range, self.temp_normal_low, temp),
            'temp_normal_high': fuzz.interp_membership(self.temp_range, self.temp_normal_high, temp),
            'temp_bahaya_low': fuzz.interp_membership(self.temp_range, self.temp_bahaya_low, temp),
            'temp_bahaya_high': fuzz.interp_membership(self.temp_range, self.temp_bahaya_high, temp),

            # TDS
            'tds_ideal': fuzz.interp_membership(self.tds_range, self.tds_ideal, tds),
            'tds_normal': fuzz.interp_membership(self.tds_range, self.tds_normal, tds),
            'tds_bahaya': fuzz.interp_membership(self.tds_range, self.tds_bahaya, tds),

            # DO
            'do_ideal': fuzz.interp_membership(self.do_range, self.do_ideal, do),
            'do_normal': fuzz.interp_membership(self.do_range, self.do_normal, do),
            'do_bahaya': fuzz.interp_membership(self.do_range, self.do_bahaya, do),
        }

        return memberships

    def apply_fuzzy_rules(self, memberships):
        # Agregasi tiap status
        # Bahaya: max dari semua bahaya
        bahaya = max(
            memberships['ph_bahaya_low'],
            memberships['ph_bahaya_high'],
            memberships['temp_bahaya_low'],
            memberships['temp_bahaya_high'],
            memberships['tds_bahaya'],
            memberships['do_bahaya']
        )
        # Normal: max dari normal rendah/tinggi
        normal = max(
            memberships['ph_normal_low'],
            memberships['ph_normal_high'],
            memberships['temp_normal_low'],
            memberships['temp_normal_high'],
            memberships['tds_normal'],
            memberships['do_normal']
        )
        # Ideal: min dari semua ideal
        ideal = min(
            memberships['ph_ideal'],
            memberships['temp_ideal'],
            memberships['tds_ideal'],
            memberships['do_ideal']
        )

        # skema prioritas: bahaya > normal > ideal
        categories = {
            'bahaya': bahaya,
            'normal': normal,
            'ideal': ideal
        }

        # Ambil status dengan nilai keanggotaan tertinggi
        # (bisa juga pakai weighted, tapi MAX cukup untuk klasifikasi)
        return max(categories, key=categories.get), categories

# ===== FEATURE ENGINEERING =====
def create_fuzzy_features(df):
    """
    Membuat fitur berbasis fuzzy membership untuk setiap sampel
    Mengembalikan: X_fuzzy, y_fuzzy, feature_names, fuzzy_system
    """
    fuzzy_system = FuzzyWaterQuality()

    # Identifikasi nama kolom yang sesuai
    ph_col = next((col for col in df.columns if 'pH' in col or 'ph' in col.lower()), None)
    temp_col = next((col for col in df.columns if 'temp' in col.lower() or 'Temp' in col), None)
    tds_col = next((col for col in df.columns if 'TDS' in col or 'tds' in col.lower()), None)
    do_col = next((col for col in df.columns if 'DO' in col or 'do' in col.lower()), None)

    print(f"\nKolom yang digunakan:")
    print(f"  pH: {ph_col}")
    print(f"  Temperature: {temp_col}")
    print(f"  TDS: {tds_col}")
    print(f"  DO: {do_col}")

    # List untuk menyimpan fitur fuzzy dan label
    fuzzy_features = []
    labels = []

    for idx, row in df.iterrows():
        ph = row[ph_col] if ph_col else 7.0
        temp = row[temp_col] if temp_col else 27.0
        tds = row[tds_col] if tds_col else 500.0
        do_val = row[do_col] if do_col else 6.0

        # Dapatkan nilai keanggotaan fuzzy
        memberships = fuzzy_system.get_membership_values(ph, temp, tds, do_val)
        # Dapatkan label menggunakan aturan fuzzy
        label, _ = fuzzy_system.apply_fuzzy_rules(memberships)

        # Gabungkan fitur asli dengan nilai keanggotaan fuzzy
        feature_vector = [
            ph, temp, tds, do_val,
            *memberships.values()
        ]

        fuzzy_features.append(feature_vector)
        labels.append(label)

    # Buat nama kolom
    feature_names = ['pH_raw', 'Temp_raw', 'TDS_raw', 'DO_raw'] + list(memberships.keys())

    return np.array(fuzzy_features), np.array(labels), feature_names, fuzzy_system

# ===== PREPROCESSING =====
print("FEATURE ENGINEERING DENGAN FUZZY LOGIC")
print("="*80)

# Jalankan fungsi untuk menghasilkan fitur fuzzy dan label
X_fuzzy, y_fuzzy, feature_names, fuzzy_system = create_fuzzy_features(df)

print(f"\nJumlah fitur: {X_fuzzy.shape[1]}")
print(f"  - Fitur asli: 4 (pH, Temp, TDS, DO)")
print(f"  - Fitur fuzzy membership: {X_fuzzy.shape[1] - 4}")

print("\nDistribusi Label dari Fuzzy Rules:")
unique, counts = np.unique(y_fuzzy, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {label}: {count} ({count/len(y_fuzzy)*100:.2f}%)")

df['status'] = y_fuzzy

In [ ]:
# ====== BALANCING DATASET SECARA DINAMIS BERBASIS MEDIAN ======
print("\n" + "="*60)
print("⚖️  MENYEIMBANGKAN DATASET (DINAMIS - BERDASARKAN MEDIAN)")
print("="*60)

feature_columns = [col for col in df.columns if col not in ['status','id','created_date','entry_id','created_at','fuzzy_score']]
X = df[feature_columns].values
y = df['status'].values
urutan_label = ['ideal', 'normal', 'bahaya']

# Tampilkan distribusi awal
print("\n📊 Distribusi Kelas SEBELUM Balancing:")
initial_distribution = {}
for label in urutan_label:
    initial_distribution[label] = np.sum(y == label)

for label in urutan_label:
    count = initial_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y):5d} sampel")

# Hitung median sebagai target jumlah sampel per kelas
jumlah_per_kelas = [initial_distribution[label] for label in urutan_label]
target_median = int(np.median(jumlah_per_kelas))

print(f"\n🎯 Median jumlah sampel per kelas: {target_median}")
print("📊 Strategi Dinamis:")
print("   - Kelas < median → oversample")
print("   - Kelas > median → undersample")
print("   - Kelas = median → pertahankan")

# Buat strategi sampling
over_strategy = {}
under_strategy = {}

for label in urutan_label:
    count = initial_distribution[label]
    if count < target_median:
        over_strategy[label] = target_median
    elif count > target_median:
        under_strategy[label] = target_median
    # Jika sama, tidak perlu sampling

# Bangun pipeline hanya jika diperlukan
steps = []

# Tambahkan oversampling jika ada kelas minority
if over_strategy:
    steps.append(('oversample', SMOTE(
        sampling_strategy=over_strategy,
        random_state=42,
        k_neighbors=min(3, min(initial_distribution.values()))  # hindari error jika sampel < 3
    )))

# Tambahkan undersampling jika ada kelas majority
if under_strategy:
    steps.append(('undersample', RandomUnderSampler(
        sampling_strategy=under_strategy,
        random_state=42
    )))

if steps:
    pipeline = Pipeline(steps)
    print("\n⏳ Proses balancing dinamis...")
    X_balanced, y_balanced = pipeline.fit_resample(X, y)
else:
    print("\n⚠️  Tidak diperlukan balancing (semua kelas sudah seimbang).")
    X_balanced, y_balanced = X.copy(), y.copy()

# Tampilkan distribusi setelah balancing
print("\n✅ Distribusi Kelas SETELAH Balancing:")
final_distribution = {}
for label in urutan_label:
    final_distribution[label] = np.sum(y_balanced == label)

for label in urutan_label:
    count = final_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y_balanced)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y_balanced):5d} sampel")

# Shuffle dataset agar random
shuffle_idx = np.random.RandomState(42).permutation(len(y_balanced))
X_balanced = X_balanced[shuffle_idx]
y_balanced = y_balanced[shuffle_idx]

print("\n🔀 Dataset telah di-shuffle untuk memastikan randomness")
print(f"✅ Dataset balanced siap digunakan: {len(y_balanced)} sampel")

# Visualisasi perbandingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before balancing
axes[0].bar(urutan_label, [initial_distribution.get(l, 0) for l in urutan_label],
            color=['green', 'orange', 'red',], alpha=0.7)
axes[0].set_title('Distribusi SEBELUM Balancing', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah Sampel', fontsize=12)
axes[0].set_xticklabels(urutan_label, rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# After balancing
axes[1].bar(urutan_label, [final_distribution.get(l, 0) for l in urutan_label],
            color=['green', 'orange', 'red'], alpha=0.7)
axes[1].set_title('Distribusi SETELAH Balancing', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Jumlah Sampel', fontsize=12)
axes[1].set_xticklabels(urutan_label, rotation=45)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
axes[1].axhline(y=target_median, color='blue', linestyle='--',
                label=f'Median Target: {target_median}', linewidth=2)
axes[1].legend()

plt.tight_layout()
plt.show()

# Update variabel X dan y dengan data yang sudah balanced
X = X_balanced
y = y_balanced

In [ ]:
# ====== 1. PERSIAPAN DATA (SETELAH BALANCING) ======
print("="*50)
print("📊 PERSIAPAN DATA - TRAIN TEST SPLIT")
print("="*50)
urutan_label = ['ideal', 'normal', 'bahaya']

# Visualisasi distribusi kelas setelah balancing
plt.figure(figsize=(8, 5))
status_counts = pd.Series(y_balanced).value_counts()
status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
colors = ['green', 'orange', 'red']
bars = plt.bar(urutan_label, status_counts_ordered, color=colors, alpha=0.7, edgecolor='black')

# Tambahkan nilai di atas bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.title('Distribusi Kelas Kualitas Air (Setelah Balancing)', fontsize=14, fontweight='bold')
plt.ylabel('Jumlah', fontsize=12)
plt.xlabel('Status Kualitas Air', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Split data menjadi training dan testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f"\n✅ Data Training: {len(X_train)} sampel ({len(X_train)/len(X_balanced)*100:.1f}%)")
print(f"✅ Data Testing: {len(X_test)} sampel ({len(X_test)/len(X_balanced)*100:.1f}%)")

# Normalisasi data menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("✅ Data berhasil dibagi dan dinormalisasi (StandardScaler)")

# ====== VALIDASI SILANG K-FOLD ======
print("\n" + "="*50)
print("🔍 TUNING HYPERPARAMETER (K-Nearest Neighbors)")
print("="*50)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_values = list(range(3, 16, 2))  # Perluas rentang pencarian
cv_scores = []
cv_stds = []

print("⏳ Mencari nilai K optimal dengan 5-Fold Cross Validation...\n")

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='euclidean')
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    mean_score = scores.mean()
    std_score = scores.std()

    cv_scores.append(mean_score)
    cv_stds.append(std_score)

    print(f"K={k:2d} | Mean CV Accuracy = {mean_score:.4f} | Std = {std_score:.4f}")

best_k = k_values[np.argmax(cv_scores)]
best_acc = max(cv_scores)

print(f"\n🎯 Nilai K Optimal: {best_k}")
print(f"🎯 Best CV Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

# ====== EVALUASI MODEL ======
print("\n" + "="*50)
print("🔍 EVALUASI MODEL (Test Set)")
print("="*50)

best_model = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='euclidean')
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)
weighted_f1 = f1_score(y_test, y_pred, average='weighted')
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n🚀 TRAINING MODEL DENGAN K={best_k}")
print(f"🎯 Test Accuracy      = {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"🎯 Weighted F1-Score  = {weighted_f1:.4f}")
print(f"🎯 Macro F1-Score     = {macro_f1:.4f}")

# Classification Report (dengan urutan label yang benar)
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, labels=urutan_label, target_names=urutan_label, zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
print("\nActual \\ Predicted:", end="")
for label in urutan_label:
    print(f"{label:>12}", end="")
print()
print("-" * 80)
for i, label in enumerate(urutan_label):
    print(f"{label:>18}", end="")
    for j in range(len(urutan_label)):
        print(f"{cm[i][j]:>12}", end="")
    print()

# VISUALISASI 2D - DATA TEST SET MENGGUNAKAN PCA
print("\n" + "="*50)
print("🎨 VISUALISASI DATA 2D (PCA)")
print("="*50)

from sklearn.decomposition import PCA

# PCA dengan 2 komponen
pca = PCA(n_components=2, random_state=42)
X_test_pca = pca.fit_transform(X_test_scaled)

# Ambil maksimal data per kelas
max_samples_per_class = 175
indices_to_plot = []
for label in urutan_label:
    mask = y_test == label
    indices = np.where(mask)[0]

    # Ambil maksimal data
    if len(indices) > max_samples_per_class:
        selected_indices = np.random.choice(indices, max_samples_per_class, replace=False)
    else:
        selected_indices = indices
    indices_to_plot.extend(selected_indices)
indices_to_plot = np.array(indices_to_plot)

# Buat figure dan axis
fig, ax = plt.subplots(figsize=(8, 7))
colors_map = {'ideal': 'green', 'normal': 'orange', 'bahaya': 'red'}

# Plot setiap kelas (hanya data terpilih)
for label in urutan_label:
    mask = y_test == label
    mask_selected = np.isin(np.arange(len(y_test)), indices_to_plot) & mask

    ax.scatter(X_test_pca[mask_selected, 0],
               X_test_pca[mask_selected, 1],
               c=colors_map[label],
               marker=markers_map[label],
               label=f'Aktual: {label}',
               alpha=0.6,
               s=100,
               edgecolors='black',
               linewidth=0.5)

# Tambahkan marker untuk prediksi yang salah (dari data terpilih)
misclassified = (y_test != y_pred) & np.isin(np.arange(len(y_test)), indices_to_plot)
if misclassified.sum() > 0:
    ax.scatter(X_test_pca[misclassified, 0],
               X_test_pca[misclassified, 1],
               marker='x', s=300, c='black', linewidth=3,
               label=f'Salah Klasifikasi ({misclassified.sum()})',
               zorder=5)

# Set label dan title
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
              fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)',
              fontsize=11, fontweight='bold')
ax.set_title(f'Visualisasi 2D Data Test Set (PCA)\nMenampilkan {len(indices_to_plot)} sampel',
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=10, framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# ====== ANALISIS LEARNING CURVE ======
print("\n" + "="*50)
print("📈 ANALISIS LEARNING CURVE")
print("="*50)

train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', color='blue', linewidth=2,
         markersize=8, label='Training Score')
plt.plot(train_sizes, test_mean, 'o-', color='green', linewidth=2,
         markersize=8, label='Cross-validation Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.15, color='blue')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std,
                 alpha=0.15, color='green')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Training Examples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - KNN Classifier (Data Balanced)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

# Analisis hasil learning curve
print("\n🔍 Analisis Learning Curve:")
gap = train_mean[-1] - test_mean[-1]
print(f"  - Training Score (final): {train_mean[-1]:.4f}")
print(f"  - CV Score (final):       {test_mean[-1]:.4f}")
print(f"  - Gap:                    {gap:.4f}")

if test_mean[-1] > 0.9 and train_mean[-1] > 0.9 and gap < 0.05:
    performance = "✅ Model terlihat SANGAT BAIK (high performance, low overfitting)"
elif gap > 0.1:
    performance = "⚠️ Model mungkin OVERFITTING (training score >> CV score)"
elif train_mean[-1] < 0.8:
    performance = "⚠️ Model mungkin UNDERFITTING (perlu model lebih kompleks)"
elif test_mean[-1] > 0.85:
    performance = "✅ Model memiliki performa BAIK dan SEIMBANG"
else:
    performance = "⚠️ Model memiliki performa CUKUP (bisa ditingkatkan)"

print(f"  Kesimpulan: {performance}")

In [ ]:
# 📁 Tentukan folder penyimpanan
output_dir = '/content/drive/MyDrive/hasil_machine_learning/3-Fuzzy-Membership-Functions/'
os.makedirs(output_dir, exist_ok=True)

# ====== SIMPAN SEMUA OUTPUT ======
print("\n" + "="*60)
print("💾 PENYIMPANAN HASIL KE GOOGLE DRIVE")
print("="*60)

# 1. Simpan Model KNN
model_path = os.path.join(output_dir, 'knn_water_quality_model.pkl')
joblib.dump(best_model, model_path)
print(f"✅ Model KNN disimpan: {model_path}")

# 2. Simpan Scaler
scaler_path = os.path.join(output_dir, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"✅ Scaler disimpan: {scaler_path}")

# 3. Simpan Feature Columns
features_path = os.path.join(output_dir, 'feature_columns.pkl')
joblib.dump(feature_columns, features_path)
print(f"✅ Feature columns disimpan: {features_path}")

# 4. Simpan Dataset dengan Status
dataset_path = os.path.join(output_dir, 'dataset_balanced_with_status.csv')
df_balanced = pd.DataFrame(X_balanced, columns=feature_columns)
numeric_columns = ['pH(ph_units)', 'Temp(cel)', 'TDS(ppm)', 'DO(mg/l)']
for col in numeric_columns:
    if col in df_balanced.columns:
        df_balanced[col] = df_balanced[col].astype(float).round(2)
df_balanced['status'] = y_balanced
df_balanced.to_csv(dataset_path, index=False)
print(f"✅ Dataset balanced disimpan: {dataset_path}")

# 5. Simpan Dataset Original dengan Status
dataset_original_path = os.path.join(output_dir, 'dataset_original_with_status.csv')
df.to_csv(dataset_original_path, index=False)
print(f"✅ Dataset original disimpan: {dataset_original_path}")

# 6. Simpan Metadata Model
metadata_path = os.path.join(output_dir, 'model_metadata.txt')
with open(metadata_path, 'w', encoding='utf-8') as f:
    f.write("="*60 + "\n")
    f.write("MODEL METADATA - KNN WATER QUALITY CLASSIFIER\n")
    f.write("="*60 + "\n\n")

    f.write("📊 INFORMASI DATASET\n")
    f.write("-" * 60 + "\n")
    f.write(f"Total Data (Original):        {len(df)} sampel\n")
    f.write(f"Total Data (After Balancing): {len(y_balanced)} sampel\n")
    f.write(f"Training Set:                 {len(y_train)} sampel ({len(y_train)/len(y_balanced)*100:.1f}%)\n")
    f.write(f"Testing Set:                  {len(y_test)} sampel ({len(y_test)/len(y_balanced)*100:.1f}%)\n")
    f.write(f"Jumlah Fitur:                 {len(feature_columns)}\n")
    f.write(f"Nama Fitur:                   {', '.join(feature_columns)}\n\n")

    f.write("🎯 Distribusi KELAS (BEFORE BALANCING):\n")
    f.write("-" * 60 + "\n")
    unique, counts = np.unique(y_fuzzy, return_counts=True)
    for label, count in zip(unique, counts) :
        f.write(f"  {label:12s}: {count:5d} ({count/len(y_fuzzy)*100:.2f}%)\n")
    f.write("\n")

    f.write("🎯 DISTRIBUSI KELAS (AFTER BALANCING)\n")
    f.write("-" * 60 + "\n")
    for label in urutan_label:
        count = np.sum(y_balanced == label)
        f.write(f"  {label:12s}: {count:5d} ({count/len(y_balanced)*100:5.2f}%)\n")
    f.write("\n")

    f.write("🔍 PARAMETER MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Algoritma:                    K-Nearest Neighbors (KNN)\n")
    f.write(f"Nilai K Optimal:              {best_k}\n")
    f.write(f"Weights:                      distance\n")
    f.write(f"Metric:                       euclidean\n")
    f.write(f"Normalisasi:                  StandardScaler\n\n")

    f.write("📈 PERFORMA MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Best CV Accuracy:             {best_acc:.4f} ({best_acc*100:.2f}%)\n")
    f.write(f"Test Accuracy:                {test_accuracy:.4f} ({test_accuracy*100:.2f}%)\n")
    f.write(f"Weighted F1-Score:            {weighted_f1:.4f}\n")
    f.write(f"Macro F1-Score:               {macro_f1:.4f}\n\n")

    f.write("📊 AKURASI PER KELAS\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
    for i, label in enumerate(urutan_label):
        class_correct = cm[i, i]
        class_total = cm[i, :].sum()
        class_accuracy = class_correct / class_total if class_total > 0 else 0
        f.write(f"  {label:12s}: {class_accuracy:.4f} ({class_accuracy*100:.2f}%) - {class_correct}/{class_total} benar\n")
    f.write("\n")

    f.write("📋 CLASSIFICATION REPORT\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    f.write(classification_report(y_test, y_pred, labels=urutan_label, target_names=urutan_label, zero_division=0))
    f.write("\n")

    f.write("🔬 CONFUSION MATRIX\n")
    f.write("-" * 60 + "\n")
    urutan_label = ['ideal', 'normal', 'bahaya']
    f.write("                              Predicted\n")
    f.write("Actual        " + "  ".join([f"{l:8s}" for l in urutan_label]) + "\n")
    for i, label in enumerate(urutan_label):
        f.write(f"{label:9s} " + "  ".join([f"{cm[i,j]:8d}" for j in range(len(urutan_label))]) + "\n")
    f.write("\n")

print(f"✅ Metadata model disimpan: {metadata_path}")

# Simpan Visualisasi
# Visualisasi 1: Distribusi Kelas Balanced
plt.figure(figsize=(8, 5))
status_counts = pd.Series(y_balanced).value_counts()
status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
colors = ['green', 'orange', 'red']
bars = plt.bar(urutan_label, status_counts_ordered, color=colors, alpha=0.7, edgecolor='black')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.title('Distribusi Kelas Kualitas Air (Setelah Balancing)', fontsize=14, fontweight='bold')
plt.ylabel('Jumlah', fontsize=12)
plt.xlabel('Status Kualitas Air', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'class_distribution_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik distribusi disimpan")

# Visualisasi 2: KNN Cross-Validation Results
plt.figure(figsize=(10, 6))
plt.errorbar(k_values, cv_scores, yerr=cv_stds, fmt='o-', capsize=5,
             color='blue', ecolor='red', linewidth=2, markersize=8)
plt.axhline(y=max(cv_scores), color='green', linestyle='--', alpha=0.7, linewidth=2)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Nilai K', fontsize=12)
plt.ylabel('Accuracy (5-Fold CV)', fontsize=12)
plt.title('Performa KNN terhadap Nilai K (Data Balanced)', fontsize=14, fontweight='bold')
plt.xticks(k_values)
plt.scatter(best_k, best_acc, color='red', s=200, zorder=5,
            label=f'Best K={best_k} (Acc={best_acc:.4f})', marker='*')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'knn_cv_results_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik CV results disimpan")

# Visualisasi 3: Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=urutan_label, yticklabels=urutan_label,
            annot_kws={"size": 14, "weight": "bold"},
            cbar_kws={'label': 'Jumlah Sampel'},
            linewidths=1, linecolor='gray')
plt.xlabel('Predicted', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=14, fontweight='bold')
plt.title('Confusion Matrix (Test Set - Data Balanced)', fontsize=16, fontweight='bold')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'confusion_matrix_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik confusion matrix disimpan")

# Visualisasi 4: Learning Curve
train_sizes_lc, train_scores_lc, test_scores_lc = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)
train_mean_lc = np.mean(train_scores_lc, axis=1)
train_std_lc = np.std(train_scores_lc, axis=1)
test_mean_lc = np.mean(test_scores_lc, axis=1)
test_std_lc = np.std(test_scores_lc, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes_lc, train_mean_lc, 'o-', color='blue', linewidth=2,
         markersize=8, label='Training Score')
plt.plot(train_sizes_lc, test_mean_lc, 'o-', color='green', linewidth=2,
         markersize=8, label='Cross-validation Score')
plt.fill_between(train_sizes_lc, train_mean_lc - train_std_lc, train_mean_lc + train_std_lc,
                 alpha=0.15, color='blue')
plt.fill_between(train_sizes_lc, test_mean_lc - test_std_lc, test_mean_lc + test_std_lc,
                 alpha=0.15, color='green')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Training Examples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - KNN Classifier (Data Balanced)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'learning_curve_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik learning curve disimpan")


# Ringkasan File yang Tersimpan
print("\n" + "="*60)
print("📁 DAFTAR FILE TERSIMPAN")
print("="*60)
saved_files = [
    ('Model KNN', 'knn_water_quality_model.pkl'),
    ('Scaler', 'scaler.pkl'),
    ('Feature Columns', 'feature_columns.pkl'),
    ('Dataset Balanced', 'dataset_balanced_with_status.csv'),
    ('Dataset Original', 'dataset_original_with_status.csv'),
    ('Metadata', 'model_metadata.txt'),
    ('Grafik Distribusi', 'class_distribution_balanced.png'),
    ('Grafik CV Results', 'knn_cv_results_balanced.png'),
    ('Grafik Confusion Matrix', 'confusion_matrix_balanced.png'),
    ('Grafik Learning Curve', 'learning_curve_balanced.png')
]

for name, filename in saved_files:
    filepath = os.path.join(output_dir, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        size_str = f"{size/1024:.2f} KB" if size < 1024*1024 else f"{size/(1024*1024):.2f} MB"
        print(f"✅ {name:25s}: {filename:40s} ({size_str})")
    else:
        print(f"❌ {name:25s}: {filename:40s} (TIDAK DITEMUKAN)")

## **BACKUP**

In [ ]:
# Load dataset dari google drive
from google.colab import drive
drive.mount('/content/drive')
# csv_path = '/content/drive/MyDrive/Colab_Notebooks/PENTING_dataset_10k.csv'
csv_path = '/content/drive/MyDrive/Colab_Notebooks/PENTING_dataset_12500rows.csv'
# csv_path = '/content/drive/MyDrive/Colab_Notebooks/PENTING_dataset_15000rows.csv'
# csv_path = '/content/drive/MyDrive/Colab_Notebooks/PENTING_dataset_17500rows.csv'
# csv_path = '/content/drive/MyDrive/Colab_Notebooks/PENTING_dataset_20000rows.csv'
df = pd.read_csv(csv_path)

print("nama file CSV:", csv_path)
print("\nInformasi Dataset:")
print(df.info())
print("\nmembaca 10 baris pertama:")
print(df.head(10))

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ====== LOAD DATASET 1: SENSOR-BASED AQUAPONICS ======
print("\nLOADING DATASET 1: Sensor-Based Aquaponics Fish Pond")
try:
    df1 = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "samahfetouh/cleaned-aquaponics-pond-dataset",
        "IoTPond3.csv"
    )
    print(f"Dataset 1 loaded successfully: {df1.shape[0]} rows, {df1.shape[1]} columns")
    print(f"   Columns: {list(df1.columns)}")
except Exception as e:
    print(f"❌ Error loading dataset 1: {e}")
    raise

# ====== LOAD DATASET 2: SMALL AQUACULTURE FISHPOND ======
print("\nLOADING DATASET 2: Small Aquaculture Fishpond")
try:
    df2 = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "bobsis/small-aquaculture-fishpond",
        "pond_iot_2023_raw.csv"
    )
    print(f"Dataset 2 loaded successfully: {df2.shape[0]} rows, {df2.shape[1]} columns")
    print(f"   Columns: {list(df2.columns)}")
except Exception as e:
    print(f"❌ Error loading dataset 2: {e}")
    raise

# ====== EKSTRAKSI DAN PENGGABUNGAN DATA ======
print("\nEXTRACTING AND MERGING DATA")
print("-"*80)

# Ekstrak kolom yang dibutuhkan dari dataset 1
df1_extracted = df1[['disolved_oxg']].copy()
df1_extracted.columns = ['DO(mg/l)']

# Ekstrak kolom dari dataset 2
df2_extracted = df2[['water_pH', 'TDS', 'water_temp']].copy()
df2_extracted.columns = ['pH(ph_units)', 'TDS(ppm)', 'Temp(cel)']

# Tentukan jumlah baris untuk dataset gabungan
n_rows_final = min(len(df1_extracted), len(df2_extracted))

# ===== STRATEGI PENGGABUNGAN =====
# Dari Dataset 1: ambil DO
df3_do = df1_extracted['DO(mg/l)'].dropna().sample(n=n_rows_final, replace=True, random_state=42).reset_index(drop=True)
# Dari Dataset 2: ambil pH, TDS, Temp
df3_ph = df2_extracted['pH(ph_units)'].dropna().sample(n=n_rows_final, replace=True, random_state=42).reset_index(drop=True)
df3_tds = df2_extracted['TDS(ppm)'].dropna().sample(n=n_rows_final, replace=True, random_state=42).reset_index(drop=True)
df3_temp = df2_extracted['Temp(cel)'].dropna().sample(n=n_rows_final, replace=True, random_state=42).reset_index(drop=True)

# ===== BUAT DATASET 3 =====
df3_combined = pd.DataFrame({
    'DO(mg/l)': df3_do,
    'pH(ph_units)': df3_ph,
    'TDS(ppm)': df3_tds,
    'Temp(cel)': df3_temp
})

print(f"Dataset 3 Created Successfully!")
print(f"    ✓ Shape: {df3_combined.shape[0]} rows × {df3_combined.shape[1]} columns")
print(f"    ✓ Columns: {list(df3_combined.columns)}")
print(f"    ✓ Missing values per column:")
for col in df3_combined.columns:
    print(f"       - {col}: {df3_combined[col].isnull().sum()}")

# ====== BATASI JUMLAH BARIS ======
if len(df3_combined) > 10000:
    df3_combined = df3_combined.sample(n=10000, random_state=42).reset_index(drop=True)
    print(f"\n✓ Data limited to {len(df3_combined)} rows")
else:
    print(f"\n✓ Data already within limit: {len(df3_combined)} rows")

# Hapus jika masih ada NaN (seharusnya tidak ada)
if df3_combined.isnull().sum().sum() > 0:
    print(f"\n⚠ Found {df3_combined.isnull().sum().sum()} missing values. Removing...")
    df3_combined = df3_combined.dropna()
    print(f"✓ After removing NaN: {len(df3_combined)} rows")
else:
    print("✓ No missing values found. Dataset is clean!\n")

# Simpan sebagai dataframe utama
df = df3_combined.copy()

print(df.info())
print(df.describe())

In [ ]:
# PRE-PROCESSING TAHAP 1: DATA CLEANING
print("TAHAP 1: DATA CLEANING")

# 1. Cek Missing Values
print("\n1. Checking Missing Values:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

if missing_values.sum() > 0:
    print(f"\nTotal missing values: {missing_values.sum()}")

    # Handling missing values
    # Opsi 1: Drop rows dengan missing values (jika sedikit)
    if missing_values.sum() < len(df) * 0.05:  # Kurang dari 5% data
        df = df.dropna()
        print("Missing values dihapus (< 5% data)")
    else:
        # Opsi 2: Impute dengan median untuk kolom numerik
        numeric_columns = df.select_dtypes(include=[np.number]).columns
        for col in numeric_columns:
            if df[col].isnull().sum() > 0:
                median_value = df[col].median()
                df[col].fillna(median_value, inplace=True)
                print(f"{col}: filled dengan median = {median_value:.2f}")
else:
    print("Tidak ada missing values")

# 2. Reset index setelah cleaning
df = df.reset_index(drop=True)

# Identifikasi kolom fitur numerik (exclude ID dan timestamp columns)
exclude_cols = ['status', 'id', 'created_date', 'entry_id', 'created_at']
numeric_features = [col for col in df.select_dtypes(include=[np.number]).columns
                   if col not in exclude_cols]

print(f"\nFitur numerik yang dianalisis: {numeric_features}")

In [ ]:
# PRE-PROCESSING TAHAP 2: FEATURE ENGINEERING
print("\nStatistical Summary (After Cleaning):")
print(df[numeric_features].describe())

# Fungsi klasifikasi
def classify_water_quality(row):
    optimal_count = 0
    danger_count = 0
    normal_count = 0

    # check pH
    if 'pH' in row or 'water_pH' in row or 'pH(ph_units)' in row:
        pH = row.get ('pH(ph_units)', row.get('water_pH'))
        if 6.7 <= pH <= 7.3:
            optimal_count += 1
        elif pH <= 6.0 or pH >= 8.0:
            danger_count += 1
        else:
            normal_count += 1

    # check temperature
    if 'temperature' in row or 'Temp(cel)' in row or 'water_temp' in row:
        temp = row.get('Temp(cel)', row.get('water_temp'))
        if 24 <= temp <= 30:
            optimal_count += 1
        elif temp <= 21 or temp >= 33:
            danger_count += 1
        else:
            normal_count += 1

    # check TDS (Total Dissolved Solids)
    if 'TDS(ppm)' in row or 'TDS' in row:
        TDS = row.get('TDS(ppm)', row.get('TDS'))
        if 200 <= TDS <= 500:
            optimal_count += 1
        elif TDS >= 1000 or TDS < 199:
            danger_count += 1
        else:
            normal_count += 1

    # check DO (Dissolved Oxygen)
    if 'do' in row or 'DO' in row or 'DO(mg/l)' in row:
        DO = row.get('DO(mg/l)', row.get('DO'))
        if DO >= 6.00:
            optimal_count += 1
        elif DO <= 3.00:
            danger_count += 1
        else:
            normal_count += 1


    # Tentukan status
    if danger_count >= 1:
        return 'bahaya'
    elif optimal_count == 4:
        return 'optimal'
    else:
        return 'normal'

# Tambahkan kolom status
df['status'] = df.apply(classify_water_quality, axis=1)

print("\nDistribusi Status:")
print(df['status'].value_counts())

In [ ]:
# MODELING
print("MODEL TRAINING")

# Dataset
X = df[numeric_features].values
y = df['status'].values
urutan_label = ['optimal', 'normal', 'bahaya']

# Split data dengan stratifikasi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nUkuran dataset:")
print(f"  - Training: {X_train.shape[0]} samples")
print(f"  - Testing: {X_test.shape[0]} samples")
print(f"  - Fitur: {X_train.shape[1]}")

# Normalisasi menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Validasi silang K-Fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

k_values = list(range(3, 12, 2))
cv_scores = []

print("\nCross-Validation Results:")
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='euclidean')
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    cv_scores.append(scores.mean())
    print(f"K={k}: Mean CV Accuracy = {scores.mean():.4f}  (Std={scores.std():.4f})")

best_k = k_values[np.argmax(cv_scores)]
print(f"\nBest K from CV = {best_k} (Mean Accuracy = {max(cv_scores):.4f})")

# Train model dengan K terbaik
best_model = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='euclidean')
best_model.fit(X_train_scaled, y_train)

# Evaluasi final di Test Set
y_pred = best_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)

print("\nHASIL EVALUASI")
print(f"\nTest Accuracy = {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print("Classification Report:")
print(classification_report(y_test, y_pred, labels=urutan_label))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=urutan_label))

In [ ]:
# SAVE MODEL & FILES
print("MENYIMPAN MODEL DAN FILE")

# Simpan model dan scaler
joblib.dump(best_model, 'knn_water_quality_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(numeric_features, 'numeric_features.pkl')
df.to_csv('dataset.csv', index=False)

# Simpan metadata preprocessing
preprocessing_info = {
    'original_shape': df.shape,
    'processed_shape': df.shape,
    'missing_values_handled': missing_values.sum(),
    'numeric_features': numeric_features,
    'best_k': best_k,
    'test_accuracy': test_accuracy,
    'scaler_type': 'StandardScaler'
}
joblib.dump(preprocessing_info, 'preprocessing_info.pkl')

print("\nMemeriksa file yang tersimpan...\n")

file_list = [
    'knn_water_quality_model.pkl',
    'scaler.pkl',
    'numeric_features.pkl',
    'preprocessing_info.pkl',
    'dataset.csv'
]

all_exist = True
for filename in file_list:
    if os.path.exists(filename):
        size = os.path.getsize(filename)
        print(f"{filename} - {size:,} bytes")
    else:
        print(f"{filename} - TIDAK DITEMUKAN!")
        all_exist = False

if all_exist:
    print("\nSemua file tersedia!")
else:
    print("\nBeberapa file tidak ditemukan!")

# Simpan ke Google Drive
print("Menghubungkan ke Google Drive...")
drive.mount('/content/drive')

# Buat folder di Google Drive
output_folder = '/content/drive/MyDrive/Model_3_kategori'
os.makedirs(output_folder, exist_ok=True)

print(f"\nMenyimpan file ke: {output_folder}")

for filename in file_list:
    if os.path.exists(filename):
        destination = os.path.join(output_folder, filename)
        shutil.copy(filename, destination)
        print(f"✅ {filename} → Google Drive")
    else:
        print(f"❌ {filename} tidak ditemukan!")

print(f"\n✅ Semua file tersimpan di Google Drive!")

In [ ]:
# Analisis statistik awal
print("\nStatistical Summary:")
print(df.describe())

# Konversi created_at ke datetime
df['created_at'] = pd.to_datetime(df['created_at'], dayfirst=True)

# 1. Preprocessing Data
# Pilih fitur numerik untuk clustering
feature_columns = ['pH(ph_units)', 'Temp(cel)', 'TDS(ppm)', 'DO(mg/l)']
X = df[feature_columns].copy()

# Handle missing values (jika ada)
X = X.fillna(X.mean())

# Normalisasi data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Principal Component Analysis (PCA)
# Melakukan PCA untuk mengurangi dimensi data
pca = PCA(n_components=0.95)  # Menjelaskan 95% variance
X_pca = pca.fit_transform(X_scaled)

print(f"\nPCA Analysis Results:")
print(f"Jumlah komponen utama yang dipilih: {pca.n_components_}")
print(f"Variance ratio per komponen: {pca.explained_variance_ratio_}")
print(f"Cumulative variance: {np.cumsum(pca.explained_variance_ratio_)}")

# Visualisasi hasil PCA
plt.figure(figsize=(10, 8))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, c='blue', edgecolors='none')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('PCA Projection of Water Quality Data')
plt.grid(True, alpha=0.3)
plt.savefig('pca_projection.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Menentukan jumlah cluster optimal
# Metode Elbow dan Silhouette Score
inertia = []
silhouette_scores = []
range_n_clusters = range(2, 8)

for n_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_pca)
    inertia.append(kmeans.inertia_)
    silhouette_avg = silhouette_score(X_pca, cluster_labels)
    silhouette_scores.append(silhouette_avg)

# Plot untuk menentukan jumlah cluster optimal
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Elbow Method
ax1.plot(range_n_clusters, inertia, 'bo-')
ax1.set_xlabel('Number of Clusters')
ax1.set_ylabel('Inertia (Within-cluster sum of squares)')
ax1.set_title('Elbow Method For Optimal k')
ax1.grid(True, alpha=0.3)

# Silhouette Score
ax2.plot(range_n_clusters, silhouette_scores, 'ro-')
ax2.set_xlabel('Number of Clusters')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score Method For Optimal k')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cluster_optimization.png', dpi=300, bbox_inches='tight')
plt.show()

# Pilih jumlah cluster optimal (berdasarkan analisis visual)
optimal_clusters = 3  # Sesuai dengan kategori ideal, normal, bahaya

# 4. Clustering dengan K-means pada data hasil PCA
kmeans = KMeans(n_clusters=optimal_clusters, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_pca)

# 5. Analisis cluster
print("\nCluster Analysis:")
# Mapping cluster centers ke feature space asli
cluster_centers_pca = kmeans.cluster_centers_
cluster_centers_scaled = pca.inverse_transform(cluster_centers_pca)
cluster_centers_original = scaler.inverse_transform(cluster_centers_scaled)

cluster_centers_df = pd.DataFrame(cluster_centers_original, columns=feature_columns)
cluster_centers_df['cluster'] = range(optimal_clusters)
print("\nCluster Centers (original feature space):")
print(cluster_centers_df)

# Analisis statistik untuk setiap cluster
for i in range(optimal_clusters):
    cluster_data = df[df['cluster'] == i]
    print(f"\nCluster {i} Statistics (Size: {len(cluster_data)}):")
    print(cluster_data[feature_columns].describe())

# 6. Visualisasi clustering dalam ruang PCA
plt.figure(figsize=(12, 9))
colors = ['green', 'orange', 'red']  # Sesuai dengan kualitas air: ideal, normal, bahaya
cluster_names = ['Cluster 0', 'Cluster 1', 'Cluster 2']

for i in range(optimal_clusters):
    cluster_points = X_pca[df['cluster'] == i]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1],
                label=f'{cluster_names[i]} (n={len(cluster_points)})',
                alpha=0.6, color=colors[i], edgecolors='none', s=30)

# Plot cluster centers
centers_pca = kmeans.cluster_centers_
plt.scatter(centers_pca[:, 0], centers_pca[:, 1],
            c='black', s=200, alpha=0.8, marker='X', label='Cluster Centers')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('PCA Clustering Results for Water Quality Data')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('pca_clustering_results.png', dpi=300, bbox_inches='tight')
plt.show()

# 7. Interpretasi cluster menjadi status kualitas air
# Berdasarkan analisis cluster centers, kita dapat memberi label pada setiap cluster
def interpret_cluster(row):
    cluster = row['cluster']

    # Analisis nilai parameter pada cluster centers untuk menentukan label
    # Cluster 0: Parameter mendekati nilai ideal
    # Cluster 1: Parameter dalam kondisi normal
    # Cluster 2: Parameter dalam kondisi bahaya

    # Sesuaikan dengan analisis cluster centers Anda
    if cluster == 0:
        return 'ideal'
    elif cluster == 1:
        return 'normal'
    else:
        return 'bahaya'

df['status_pca'] = df.apply(interpret_cluster, axis=1)

# 8. Perbandingan dengan metode rule-based sebelumnya
# Implementasi singkat rule-based untuk perbandingan
def classify_water_quality_rule(row):
    prioritas_order = ['bahaya', 'normal', 'ideal']
    kondisi_terbaik = 'ideal'

    # Check pH
    pH = row['pH(ph_units)']
    if pd.notna(pH):
        if pH < 6.0 or pH > 8.0:
            kondisi = 'bahaya'
        elif (6.0 <= pH < 6.8) or (7.2 < pH <= 8.0):
            kondisi = 'normal'
        elif 6.8 <= pH <= 7.2:
            kondisi = 'ideal'
        else:
            kondisi = 'normal'
        if prioritas_order.index(kondisi) < prioritas_order.index(kondisi_terbaik):
            kondisi_terbaik = kondisi

    # Check suhu
    temp = row['Temp(cel)']
    if pd.notna(temp):
        if temp < 22.0 or temp > 32.0:
            kondisi = 'bahaya'
        elif (22.0 <= temp < 25.0) or (29.0 < temp <= 32.0):
            kondisi = 'normal'
        elif 25.0 <= temp <= 29.0:
            kondisi = 'ideal'
        else:
            kondisi = 'normal'
        if prioritas_order.index(kondisi) < prioritas_order.index(kondisi_terbaik):
            kondisi_terbaik = kondisi

    # Check TDS
    TDS = row['TDS(ppm)']
    if pd.notna(TDS):
        if TDS > 1000:
            kondisi = 'bahaya'
        elif 401 <= TDS <= 1000:
            kondisi = 'normal'
        elif 150 <= TDS <= 400:
            kondisi = 'ideal'
        else:
            kondisi = 'normal'
        if prioritas_order.index(kondisi) < prioritas_order.index(kondisi_terbaik):
            kondisi_terbaik = kondisi

    # Check DO
    DO = row['DO(mg/l)']
    if pd.notna(DO):
        if DO < 3.00:
            kondisi = 'bahaya'
        elif 3.00 <= DO <= 6.00:
            kondisi = 'normal'
        elif DO > 6.00:
            kondisi = 'ideal'
        else:
            kondisi = 'normal'
        if prioritas_order.index(kondisi) < prioritas_order.index(kondisi_terbaik):
            kondisi_terbaik = kondisi

    return kondisi_terbaik

df['status_rule'] = df.apply(classify_water_quality_rule, axis=1)

# 9. Analisis perbandingan kedua metode
print("\nPerbandingan Distribusi Status:")
print("\nPCA Clustering Method:")
print(df['status_pca'].value_counts())
print("\nRule-based Method:")
print(df['status_rule'].value_counts())
print("\nKontingensi Perbandingan Metode:")

# Visualisasi perbandingan distribusi
plt.figure(figsize=(12, 6))
ax = plt.subplot(111)

# Plot distribusi kedua metode
bar_width = 0.35
status_order = ['ideal', 'normal', 'bahaya']
x = np.arange(len(status_order))

rule_counts = [df['status_rule'].value_counts().get(s, 0) for s in status_order]
pca_counts = [df['status_pca'].value_counts().get(s, 0) for s in status_order]

bar1 = plt.bar(x - bar_width/2, rule_counts, bar_width, label='Rule-based', color='skyblue', edgecolor='black')
bar2 = plt.bar(x + bar_width/2, pca_counts, bar_width, label='PCA Clustering', color='salmon', edgecolor='black')

plt.xlabel('Status Kualitas Air')
plt.ylabel('Jumlah Data')
plt.title('Perbandingan Distribusi Status: Rule-based vs PCA Clustering')
plt.xticks(x, status_order)
plt.legend()

# Tambahkan nilai di atas bar
for i, (rect1, rect2) in enumerate(zip(bar1, bar2)):
    height1 = rect1.get_height()
    height2 = rect2.get_height()
    ax.annotate(f'{height1}',
                xy=(rect1.get_x() + rect1.get_width() / 2, height1),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom')
    ax.annotate(f'{height2}',
                xy=(rect2.get_x() + rect2.get_width() / 2, height2),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('comparison_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# 10. Analisis tambahan: Visualisasi boxplot parameter per cluster
plt.figure(figsize=(15, 10))
for i, feature in enumerate(feature_columns):
    plt.subplot(2, 2, i+1)
    sns.boxplot(x='cluster', y=feature, data=df, palette=colors)
    plt.title(f'Distribusi {feature} per Cluster')
    plt.xlabel('Cluster')
    plt.ylabel(feature)

plt.tight_layout()
plt.savefig('cluster_feature_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# 11. Simpan hasil
output_df = df.copy()
output_df['created_at'] = output_df['created_at'].dt.strftime('%d/%m/%Y %H:%M')
output_df.to_csv('water_quality_pca_clustering_results.csv', index=False)
print("\nHasil clustering telah disimpan ke 'water_quality_pca_clustering_results.csv'")

## **TERBARU FUZZY LOGIC**

In [ ]:
# ===== PREPROCESSING =====
print("\n" + "="*80)
print("🚀 STARTING MODEL BUILDING PROCESS")
print("="*80)

# FUZZY MEMBERSHIP FUNCTIONS
class FuzzyWaterQuality:
    """
    Kelas untuk mendefinisikan fungsi keanggotaan fuzzy
    dengan 3 status: ideal, normal, bahaya
    """
    def __init__(self):
        # Definisi range untuk setiap parameter
        self.do_range = np.arange(0, 20.1, 0.1)
        self.ph_range = np.arange(0, 14.1, 0.1)
        self.tds_range = np.arange(0, 2000.1, 1)
        self.temp_range = np.arange(0, 50.1, 0.1)

        # Setup fuzzy membership functions
        self._setup_do_fuzzy()
        self._setup_ph_fuzzy()
        self._setup_tds_fuzzy()
        self._setup_temp_fuzzy()

    def _setup_do_fuzzy(self):
        """Fungsi keanggotaan fuzzy untuk DO """
        self.do_bahaya = fuzz.trapmf(self.do_range, [0, 0, 2.5, 3.0])
        self.do_normal = fuzz.trimf(self.do_range, [2.8, 4.0, 5.2])
        self.do_ideal = fuzz.trapmf(self.do_range, [5.0, 6.0, 20, 20])

    def _setup_ph_fuzzy(self):
        """Fungsi keanggotaan fuzzy untuk pH"""
        self.ph_bahaya_low = fuzz.trapmf(self.ph_range, [0, 0, 5.5, 5.9])
        self.ph_bahaya_high = fuzz.trapmf(self.ph_range, [7.9, 8.3, 14, 14])
        self.ph_normal_low = fuzz.trimf(self.ph_range, [5.8, 6.2, 6.6])
        self.ph_normal_high = fuzz.trimf(self.ph_range, [7.3, 7.65, 8.0])
        self.ph_ideal = fuzz.trimf(self.ph_range, [6.5, 6.95, 7.4])

    def _setup_tds_fuzzy(self):
        """Fungsi keanggotaan fuzzy untuk TDS"""
        self.tds_ideal = fuzz.trimf(self.tds_range, [1, 200, 400])
        self.tds_normal = fuzz.trimf(self.tds_range, [380, 500, 900])
        self.tds_bahaya = fuzz.trapmf(self.tds_range, [850, 1000, 2000, 2000])

    def _setup_temp_fuzzy(self):
        """Fungsi keanggotaan fuzzy untuk Suhu"""
        self.temp_bahaya_low = fuzz.trapmf(self.temp_range, [0, 0, 20.0, 22.5])
        self.temp_bahaya_high = fuzz.trapmf(self.temp_range, [32.0, 34.0, 50.0, 50.0])
        self.temp_normal_low = fuzz.trimf(self.temp_range, [22.0, 23.5, 25.0])
        self.temp_normal_high = fuzz.trimf(self.temp_range, [29.5, 31.0, 32.5])
        self.temp_ideal = fuzz.trimf(self.temp_range, [24.5, 27.5, 30.0])

    def get_membership_values(self, ph, temp, tds, do):
        """Menghitung nilai keanggotaan untuk semua kategori"""
        do = np.clip(do, self.do_range[0], self.do_range[-1])
        ph = np.clip(ph, self.ph_range[0], self.ph_range[-1])
        tds = np.clip(tds, self.tds_range[0], self.tds_range[-1])
        temp = np.clip(temp, self.temp_range[0], self.temp_range[-1])

        memberships = {
            # DO
            'do_ideal': fuzz.interp_membership(self.do_range, self.do_ideal, do),
            'do_normal': fuzz.interp_membership(self.do_range, self.do_normal, do),
            'do_bahaya': fuzz.interp_membership(self.do_range, self.do_bahaya, do),

            # pH
            'ph_ideal': fuzz.interp_membership(self.ph_range, self.ph_ideal, ph),
            'ph_normal_low': fuzz.interp_membership(self.ph_range, self.ph_normal_low, ph),
            'ph_normal_high': fuzz.interp_membership(self.ph_range, self.ph_normal_high, ph),
            'ph_bahaya_low': fuzz.interp_membership(self.ph_range, self.ph_bahaya_low, ph),
            'ph_bahaya_high': fuzz.interp_membership(self.ph_range, self.ph_bahaya_high, ph),

            # TDS
            'tds_ideal': fuzz.interp_membership(self.tds_range, self.tds_ideal, tds),
            'tds_normal': fuzz.interp_membership(self.tds_range, self.tds_normal, tds),
            'tds_bahaya': fuzz.interp_membership(self.tds_range, self.tds_bahaya, tds),

            # Temperature
            'temp_ideal': fuzz.interp_membership(self.temp_range, self.temp_ideal, temp),
            'temp_normal_low': fuzz.interp_membership(self.temp_range, self.temp_normal_low, temp),
            'temp_normal_high': fuzz.interp_membership(self.temp_range, self.temp_normal_high, temp),
            'temp_bahaya_low': fuzz.interp_membership(self.temp_range, self.temp_bahaya_low, temp),
            'temp_bahaya_high': fuzz.interp_membership(self.temp_range, self.temp_bahaya_high, temp),
        }

        return memberships

    def apply_fuzzy_rules(self, memberships):
        # Agregasi tiap status
        # Bahaya: max dari semua bahaya
        bahaya = max(
            memberships['ph_bahaya_low'],
            memberships['ph_bahaya_high'],
            memberships['temp_bahaya_low'],
            memberships['temp_bahaya_high'],
            memberships['tds_bahaya'],
            memberships['do_bahaya']
        )
        # Normal: max dari normal rendah/tinggi
        normal = max(
            memberships['ph_normal_low'],
            memberships['ph_normal_high'],
            memberships['temp_normal_low'],
            memberships['temp_normal_high'],
            memberships['tds_normal'],
            memberships['do_normal']
        )
        # Ideal: min dari semua ideal
        ideal = min(
            memberships['ph_ideal'],
            memberships['temp_ideal'],
            memberships['tds_ideal'],
            memberships['do_ideal']
        )

        categories = {
            'bahaya': bahaya,
            'normal': normal,
            'ideal': ideal
        }

        return max(categories, key=categories.get), categories

# ===== FEATURE ENGINEERING =====
def create_fuzzy_features(df):
    """
    Membuat fitur berbasis fuzzy membership untuk setiap sampel
    """
    fuzzy_system = FuzzyWaterQuality()

    # Identifikasi nama kolom
    do_col = 'DO(mg/l)'
    ph_col = 'pH(ph_units)'
    tds_col = 'TDS(ppm)'
    temp_col = 'Temp(cel)'

    print(f"\nKolom yang digunakan:")
    print(f"  DO: {do_col}")
    print(f"  pH: {ph_col}")
    print(f"  TDS: {tds_col}")
    print(f"  Temperature: {temp_col}")

    fuzzy_features = []
    labels = []

    for idx, row in df.iterrows():
        do_val = row[do_col]
        ph = row[ph_col]
        tds = row[tds_col]
        temp = row[temp_col]

        memberships = fuzzy_system.get_membership_values(ph, temp, tds, do_val)
        label, _ = fuzzy_system.apply_fuzzy_rules(memberships)

        feature_vector = [
            do_val, ph, tds, temp,
            *memberships.values()
        ]

        fuzzy_features.append(feature_vector)
        labels.append(label)

    feature_names = ['DO_raw', 'pH_raw', 'TDS_raw', 'Temp_raw'] + list(memberships.keys())

    return np.array(fuzzy_features), np.array(labels), feature_names, fuzzy_system

X_fuzzy, y_fuzzy, feature_names, fuzzy_system = create_fuzzy_features(df)

print(f"\nJumlah fitur: {X_fuzzy.shape[1]}")
print(f"  - Fitur asli: 4 (DO, pH, TDS, Temp)")
print(f"  - Fitur fuzzy membership: {X_fuzzy.shape[1] - 4}")

print("\nDistribusi Label dari Fuzzy Rules:")
unique, counts = np.unique(y_fuzzy, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {label}: {count} ({count/len(y_fuzzy)*100:.2f}%)")

df['status'] = y_fuzzy

In [ ]:
# ====== BALANCING DATASET SECARA DINAMIS BERBASIS MEDIAN ======
print("\n" + "="*60)
print("⚖️ MENYEIMBANGKAN DATASET (DINAMIS - BERDASARKAN MEDIAN)")
print("="*60)

feature_columns = [col for col in df.columns if col not in ['status','id','created_date','entry_id','created_at','fuzzy_score']]
X = df[feature_columns].values
y = df['status'].values
urutan_label = ['ideal', 'normal', 'bahaya']

# Tampilkan distribusi awal
print("\n📊 Distribusi Kelas SEBELUM Balancing:")
initial_distribution = {}
for label in urutan_label:
    initial_distribution[label] = np.sum(y == label)

for label in urutan_label:
    count = initial_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y):5d} sampel")

# Hitung median sebagai target jumlah sampel per kelas
jumlah_per_kelas = [initial_distribution[label] for label in urutan_label]
target_median = int(np.median(jumlah_per_kelas))

print(f"\n🎯 Median jumlah sampel per kelas: {target_median}")
print("📊 Strategi Dinamis:")
print("   - Kelas < median → oversample")
print("   - Kelas > median → undersample")
print("   - Kelas = median → pertahankan")

# Buat strategi sampling
over_strategy = {}
under_strategy = {}

for label in urutan_label:
    count = initial_distribution[label]
    if count < target_median:
        over_strategy[label] = target_median
    elif count > target_median:
        under_strategy[label] = target_median

# Bangun pipeline hanya jika diperlukan
steps = []

# Tambahkan oversampling jika ada kelas minority
if over_strategy:
    steps.append(('oversample', SMOTE(
        sampling_strategy=over_strategy,
        random_state=42,
        k_neighbors=min(3, min(initial_distribution.values()) - 1)
    )))

# Tambahkan undersampling jika ada kelas majority
if under_strategy:
    steps.append(('undersample', RandomUnderSampler(
        sampling_strategy=under_strategy,
        random_state=42
    )))

if steps:
    pipeline = Pipeline(steps)
    print("\n⏳ Proses balancing dinamis...")
    X_balanced, y_balanced = pipeline.fit_resample(X, y)
else:
    print("\n⚠️ Tidak diperlukan balancing (semua kelas sudah seimbang).")
    X_balanced, y_balanced = X.copy(), y.copy()

# Tampilkan distribusi setelah balancing
print("\n✅ Distribusi Kelas SETELAH Balancing:")
final_distribution = {}
for label in urutan_label:
    final_distribution[label] = np.sum(y_balanced == label)

for label in urutan_label:
    count = final_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y_balanced)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y_balanced):5d} sampel")

# Shuffle dataset agar random
shuffle_idx = np.random.RandomState(42).permutation(len(y_balanced))
X_balanced = X_balanced[shuffle_idx]
y_balanced = y_balanced[shuffle_idx]

print("\n🔀 Dataset telah di-shuffle untuk memastikan randomness")
print(f"✅ Dataset balanced siap digunakan: {len(y_balanced)} sampel")

# Visualisasi perbandingan
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Before balancing
axes[0].bar(urutan_label, [initial_distribution.get(l, 0) for l in urutan_label],
            color=['green', 'orange', 'red'], alpha=0.7)
axes[0].set_title('Distribusi SEBELUM Balancing', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Jumlah Sampel', fontsize=12)
axes[0].set_xticklabels(urutan_label, rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# After balancing
axes[1].bar(urutan_label, [final_distribution.get(l, 0) for l in urutan_label],
            color=['green', 'orange', 'red'], alpha=0.7)
axes[1].set_title('Distribusi SETELAH Balancing', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Jumlah Sampel', fontsize=12)
axes[1].set_xticklabels(urutan_label, rotation=45)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
axes[1].axhline(y=target_median, color='blue', linestyle='--',
                label=f'Median Target: {target_median}', linewidth=2)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Update variabel X dan y dengan data yang sudah balanced
X = X_balanced
y = y_balanced

# ====== 1. PERSIAPAN DATA (SETELAH BALANCING) ======
print("="*50)
print("📊 PERSIAPAN DATA - TRAIN TEST SPLIT")
print("="*50)
urutan_label = ['ideal', 'normal', 'bahaya']

# Visualisasi distribusi kelas setelah balancing
plt.figure(figsize=(6, 4))
status_counts = pd.Series(y_balanced).value_counts()
status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
colors = ['green', 'orange', 'red']
bars = plt.bar(urutan_label, status_counts_ordered, color=colors, alpha=0.7, edgecolor='black')

# Tambahkan nilai di atas bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.title('Distribusi Kelas Kualitas Air (Setelah Balancing)', fontsize=12, fontweight='bold')
plt.ylabel('Jumlah', fontsize=12)
plt.xlabel('Status Kualitas Air', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Split data menjadi training dan testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f"\n✅ Data Training: {len(X_train)} sampel ({len(X_train)/len(X_balanced)*100:.1f}%)")
print(f"✅ Data Testing: {len(X_test)} sampel ({len(X_test)/len(X_balanced)*100:.1f}%)")

# Normalisasi data menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("✅ Data berhasil dibagi dan dinormalisasi (StandardScaler)")

# ====== VALIDASI SILANG K-FOLD ======
print("\n" + "="*50)
print("🔍 TUNING HYPERPARAMETER (K-Nearest Neighbors)")
print("="*50)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_values = list(range(3, 16, 2))
cv_scores = []
cv_stds = []

print("⏳ Mencari nilai K optimal dengan 5-Fold Cross Validation...\n")

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='euclidean')
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    mean_score = scores.mean()
    std_score = scores.std()

    cv_scores.append(mean_score)
    cv_stds.append(std_score)

    print(f"K={k:2d} | Mean CV Accuracy = {mean_score:.4f} | Std = {std_score:.4f}")

best_k = k_values[np.argmax(cv_scores)]
best_acc = max(cv_scores)

print(f"\n🎯 Nilai K Optimal: {best_k}")
print(f"🎯 Best CV Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

# ====== EVALUASI MODEL ======
print("\n" + "="*50)
print("🔍 EVALUASI MODEL (Test Set)")
print("="*50)

best_model = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='euclidean')
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)
weighted_f1 = f1_score(y_test, y_pred, average='weighted')
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n🚀 TRAINING MODEL DENGAN K={best_k}")
print(f"🎯 Test Accuracy      = {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"🎯 Weighted F1-Score  = {weighted_f1:.4f}")
print(f"🎯 Macro F1-Score     = {macro_f1:.4f}")

# Classification Report
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, labels=urutan_label, target_names=urutan_label, zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
print("\nActual \\ Predicted:", end="")
for label in urutan_label:
    print(f"{label:>12}", end="")
print()
print("-" * 80)
for i, label in enumerate(urutan_label):
    print(f"{label:>18}", end="")
    for j in range(len(urutan_label)):
        print(f"{cm[i][j]:>12}", end="")
    print()

# VISUALISASI 2D - DATA TEST SET MENGGUNAKAN PCA
print("\n" + "="*50)
print("🎨 VISUALISASI DATA 2D (PCA)")
print("="*50)

from sklearn.decomposition import PCA

# PCA dengan 2 komponen
pca = PCA(n_components=2, random_state=42)
X_test_pca = pca.fit_transform(X_test_scaled)

# Ambil maksimal data per kelas
max_samples_per_class = 167
indices_to_plot = []
markers_map = {'ideal': 'o', 'normal': 's', 'bahaya': '^'}
for label in urutan_label:
    mask = y_test == label
    indices = np.where(mask)[0]

    # Ambil maksimal data
    if len(indices) > max_samples_per_class:
        selected_indices = np.random.choice(indices, max_samples_per_class, replace=False)
    else:
        selected_indices = indices
    indices_to_plot.extend(selected_indices)
indices_to_plot = np.array(indices_to_plot)

# Buat figure dan axis
fig, ax = plt.subplots(figsize=(6, 6))
colors_map = {'ideal': 'green', 'normal': 'orange', 'bahaya': 'red'}

# Plot setiap kelas (hanya data terpilih)
for label in urutan_label:
    mask = y_test == label
    mask_selected = np.isin(np.arange(len(y_test)), indices_to_plot) & mask

    ax.scatter(X_test_pca[mask_selected, 0],
               X_test_pca[mask_selected, 1],
               c=colors_map[label],
               marker=markers_map[label],
               label=f'Aktual: {label}',
               alpha=0.6,
               s=100,
               edgecolors='black',
               linewidth=0.5)

# Tambahkan marker untuk prediksi yang salah
misclassified = (y_test != y_pred) & np.isin(np.arange(len(y_test)), indices_to_plot)
if misclassified.sum() > 0:
    ax.scatter(X_test_pca[misclassified, 0],
               X_test_pca[misclassified, 1],
               marker='x', s=300, c='black', linewidth=3,
               label=f'Salah Klasifikasi ({misclassified.sum()})',
               zorder=5)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
              fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)',
              fontsize=11, fontweight='bold')
ax.set_title(f'Visualisasi 2D Data Test Set (PCA)\nMenampilkan {len(indices_to_plot)} sampel',
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=10, framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# ====== ANALISIS LEARNING CURVE ======
print("\n" + "="*50)
print("📈 ANALISIS LEARNING CURVE")
print("="*50)

train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', color='blue', linewidth=2,
         markersize=8, label='Training Score')
plt.plot(train_sizes, test_mean, 'o-', color='green', linewidth=2,
         markersize=8, label='Cross-validation Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.15, color='blue')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std,
                 alpha=0.15, color='green')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Training Examples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - KNN Classifier (Data Balanced)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

# Analisis hasil learning curve
print("\n🔍 Analisis Learning Curve:")
gap = train_mean[-1] - test_mean[-1]
print(f"  - Training Score (final): {train_mean[-1]:.4f}")
print(f"  - CV Score (final):       {test_mean[-1]:.4f}")
print(f"  - Gap:                    {gap:.4f}")

if test_mean[-1] > 0.9 and train_mean[-1] > 0.9 and gap < 0.05:
    performance = "✅ Model terlihat SANGAT BAIK (high performance, low overfitting)"
elif gap > 0.1:
    performance = "⚠️ Model mungkin OVERFITTING (training score >> CV score)"
elif train_mean[-1] < 0.8:
    performance = "⚠️ Model mungkin UNDERFITTING (perlu model lebih kompleks)"
elif test_mean[-1] > 0.85:
    performance = "✅ Model memiliki performa BAIK dan SEIMBANG"
else:
    performance = "⚠️ Model memiliki performa CUKUP (bisa ditingkatkan)"

print(f"  Kesimpulan: {performance}")

In [ ]:
# 🗂️ Tentukan folder penyimpanan
output_dir = '/content/drive/MyDrive/3-Fuzzy-logic-part1/'
os.makedirs(output_dir, exist_ok=True)

# ====== SIMPAN SEMUA OUTPUT ======
print("\n" + "="*60)
print("💾 PENYIMPANAN HASIL KE GOOGLE DRIVE")
print("="*60)

# 1. Simpan Model KNN
model_path = os.path.join(output_dir, 'knn_water_quality_model.pkl')
joblib.dump(best_model, model_path)
print(f"✅ Model KNN disimpan: {model_path}")

# 2. Simpan Scaler
scaler_path = os.path.join(output_dir, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"✅ Scaler disimpan: {scaler_path}")

# 3. Simpan Feature Columns
features_path = os.path.join(output_dir, 'feature_columns.pkl')
joblib.dump(feature_columns, features_path)
print(f"✅ Feature columns disimpan: {features_path}")

# 4. Simpan Dataset dengan Status
dataset_path = os.path.join(output_dir, 'dataset_balanced_with_status.csv')
df_balanced = pd.DataFrame(X_balanced, columns=feature_columns)
numeric_columns = ['pH(ph_units)', 'Temp(cel)', 'TDS(ppm)', 'DO(mg/l)']
for col in numeric_columns:
    if col in df_balanced.columns:
        df_balanced[col] = df_balanced[col].astype(float).round(2)
df_balanced['status'] = y_balanced
df_balanced.to_csv(dataset_path, index=False)
print(f"✅ Dataset balanced disimpan: {dataset_path}")

# 5. Simpan Dataset Original dengan Status
dataset_original_path = os.path.join(output_dir, 'dataset_original_with_status.csv')
df.to_csv(dataset_original_path, index=False)
print(f"✅ Dataset original disimpan: {dataset_original_path}")

# 6. Simpan Metadata Model
metadata_path = os.path.join(output_dir, 'model_metadata.txt')
with open(metadata_path, 'w', encoding='utf-8') as f:
    f.write("="*60 + "\n")
    f.write("MODEL METADATA - KNN WATER QUALITY CLASSIFIER\n")
    f.write("="*60 + "\n\n")

    f.write("📊 INFORMASI DATASET\n")
    f.write("-" * 60 + "\n")
    f.write(f"Total Data (Original):        {len(df)} sampel\n")
    f.write(f"Total Data (After Balancing): {len(y_balanced)} sampel\n")
    f.write(f"Training Set:                 {len(y_train)} sampel ({len(y_train)/len(y_balanced)*100:.1f}%)\n")
    f.write(f"Testing Set:                  {len(y_test)} sampel ({len(y_test)/len(y_balanced)*100:.1f}%)\n")
    f.write(f"Jumlah Fitur:                 {len(feature_columns)}\n")
    f.write(f"Nama Fitur:                   {', '.join(feature_columns)}\n\n")

    f.write("🎯 Distribusi KELAS (BEFORE BALANCING):\n")
    f.write("-" * 60 + "\n")
    unique, counts = np.unique(y_fuzzy, return_counts=True)
    for label, count in zip(unique, counts):
        f.write(f"  {label:12s}: {count:5d} ({count/len(y_fuzzy)*100:.2f}%)\n")
    f.write("\n")

    f.write("🎯 DISTRIBUSI KELAS (AFTER BALANCING)\n")
    f.write("-" * 60 + "\n")
    for label in urutan_label:
        count = np.sum(y_balanced == label)
        f.write(f"  {label:12s}: {count:5d} ({count/len(y_balanced)*100:5.2f}%)\n")
    f.write("\n")

    f.write("🔍 PARAMETER MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Algoritma:                    K-Nearest Neighbors (KNN)\n")
    f.write(f"Nilai K Optimal:              {best_k}\n")
    f.write(f"Weights:                      distance\n")
    f.write(f"Metric:                       euclidean\n")
    f.write(f"Normalisasi:                  StandardScaler\n\n")

    f.write("📈 PERFORMA MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Best CV Accuracy:             {best_acc:.4f} ({best_acc*100:.2f}%)\n")
    f.write(f"Test Accuracy:                {test_accuracy:.4f} ({test_accuracy*100:.2f}%)\n")
    f.write(f"Weighted F1-Score:            {weighted_f1:.4f}\n")
    f.write(f"Macro F1-Score:               {macro_f1:.4f}\n\n")

    f.write("📊 AKURASI PER KELAS\n")
    f.write("-" * 60 + "\n")
    cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
    for i, label in enumerate(urutan_label):
        class_correct = cm[i, i]
        class_total = cm[i, :].sum()
        class_accuracy = class_correct / class_total if class_total > 0 else 0
        f.write(f"  {label:12s}: {class_accuracy:.4f} ({class_accuracy*100:.2f}%) - {class_correct}/{class_total} benar\n")
    f.write("\n")

    f.write("📋 CLASSIFICATION REPORT\n")
    f.write("-" * 60 + "\n")
    f.write(classification_report(y_test, y_pred, labels=urutan_label, target_names=urutan_label, zero_division=0))
    f.write("\n")

    f.write("🔬 CONFUSION MATRIX\n")
    f.write("-" * 60 + "\n")
    f.write("                              Predicted\n")
    f.write("Actual        " + "  ".join([f"{l:8s}" for l in urutan_label]) + "\n")
    for i, label in enumerate(urutan_label):
        f.write(f"{label:9s} " + "  ".join([f"{cm[i,j]:8d}" for j in range(len(urutan_label))]) + "\n")
    f.write("\n")

print(f"✅ Metadata model disimpan: {metadata_path}")

# Simpan Visualisasi
# Visualisasi 1: Distribusi Kelas Balanced
plt.figure(figsize=(8, 5))
status_counts = pd.Series(y_balanced).value_counts()
status_counts_ordered = status_counts.reindex(urutan_label, fill_value=0)
colors = ['green', 'orange', 'red']
bars = plt.bar(urutan_label, status_counts_ordered, color=colors, alpha=0.7, edgecolor='black')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.title('Distribusi Kelas Kualitas Air (Setelah Balancing)', fontsize=14, fontweight='bold')
plt.ylabel('Jumlah', fontsize=12)
plt.xlabel('Status Kualitas Air', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'class_distribution_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik distribusi disimpan")

# Visualisasi 2: KNN Cross-Validation Results
plt.figure(figsize=(10, 6))
plt.errorbar(k_values, cv_scores, yerr=cv_stds, fmt='o-', capsize=5,
             color='blue', ecolor='red', linewidth=2, markersize=8)
plt.axhline(y=max(cv_scores), color='green', linestyle='--', alpha=0.7, linewidth=2)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Nilai K', fontsize=12)
plt.ylabel('Accuracy (5-Fold CV)', fontsize=12)
plt.title('Performa KNN terhadap Nilai K (Data Balanced)', fontsize=14, fontweight='bold')
plt.xticks(k_values)
plt.scatter(best_k, best_acc, color='red', s=200, zorder=5,
            label=f'Best K={best_k} (Acc={best_acc:.4f})', marker='*')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'knn_cv_results_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik CV results disimpan")

# Visualisasi 3: Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=urutan_label, yticklabels=urutan_label,
            annot_kws={"size": 14, "weight": "bold"},
            cbar_kws={'label': 'Jumlah Sampel'},
            linewidths=1, linecolor='gray')
plt.xlabel('Predicted', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=14, fontweight='bold')
plt.title('Confusion Matrix (Test Set - Data Balanced)', fontsize=16, fontweight='bold')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'confusion_matrix_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik confusion matrix disimpan")

# Visualisasi 4: Learning Curve
train_sizes_lc, train_scores_lc, test_scores_lc = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)
train_mean_lc = np.mean(train_scores_lc, axis=1)
train_std_lc = np.std(train_scores_lc, axis=1)
test_mean_lc = np.mean(test_scores_lc, axis=1)
test_std_lc = np.std(test_scores_lc, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes_lc, train_mean_lc, 'o-', color='blue', linewidth=2,
         markersize=8, label='Training Score')
plt.plot(train_sizes_lc, test_mean_lc, 'o-', color='green', linewidth=2,
         markersize=8, label='Cross-validation Score')
plt.fill_between(train_sizes_lc, train_mean_lc - train_std_lc, train_mean_lc + train_std_lc,
                 alpha=0.15, color='blue')
plt.fill_between(train_sizes_lc, test_mean_lc - test_std_lc, test_mean_lc + test_std_lc,
                 alpha=0.15, color='green')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Training Examples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - KNN Classifier (Data Balanced)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'learning_curve_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik learning curve disimpan")

# Ringkasan File yang Tersimpan
print("\n" + "="*60)
print("🗂️ DAFTAR FILE TERSIMPAN")
print("="*60)
saved_files = [
    ('Model KNN', 'knn_water_quality_model.pkl'),
    ('Scaler', 'scaler.pkl'),
    ('Feature Columns', 'feature_columns.pkl'),
    ('Dataset Balanced', 'dataset_balanced_with_status.csv'),
    ('Dataset Original', 'dataset_original_with_status.csv'),
    ('Metadata', 'model_metadata.txt'),
    ('Grafik Distribusi', 'class_distribution_balanced.png'),
    ('Grafik CV Results', 'knn_cv_results_balanced.png'),
    ('Grafik Confusion Matrix', 'confusion_matrix_balanced.png'),
    ('Grafik Learning Curve', 'learning_curve_balanced.png')
]

for name, filename in saved_files:
    filepath = os.path.join(output_dir, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        size_str = f"{size/1024:.2f} KB" if size < 1024*1024 else f"{size/(1024*1024):.2f} MB"
        print(f"✅ {name:25s}: {filename:40s} ({size_str})")
    else:
        print(f"❌ {name:25s}: {filename:40s} (TIDAK DITEMUKAN)")

print("\n" + "="*60)
print("✅ PROSES SELESAI!")
print("="*60)